# Indian airlines after COVID: performance index and dashboard
Final dashboard notebook. Run it after the download and cleaning notebooks. It **reads** `data/clean/` and `data/interim/`, downloads the city-pair file once into `data/raw/github/` if it is not there (from the GitHub commit the collection notebook recorded), and **writes one file**: `outputs/dashboard/dgca_performance_dashboard_<date>.html` (self-contained, opens offline).

**Order of the page:** the ten metrics and how each is calculated, then the scorecard, then one section per pillar (recovery, operations, momentum), the route network from the city-pair data, international flights, and finally relationships, prediction and conclusions.

**What changed from the previous version**
- Countermeasures for new airlines: log growth, launch months left out of volatility, momentum damped for airlines under 36 months old, robust (median-based) scaling with minimum spreads, and a neutral 50 for a missing pillar. A chart compares the old and new index.
- Metrics now use passenger-km for recovery, growth, share and volatility; passengers stay in the recovery status because the project question is about passenger numbers.
- The 3D landscape is replaced by a stacked area on one time axis; parallel coordinates and the weighting slope chart are removed (the weighting result is now one sentence).
- Correlation matrix: in the HTML page, clicking any cell draws that pair of metrics.
- The distance between airlines is shown and explained.
- New sections for the city-pair data (8 charts) and international flights (2 charts).

Libraries: pandas, numpy, plotly, scipy, pyarrow, requests.

## 1. Setup and data

In [1]:
# SETUP: paths, libraries and data. This notebook READS data/ (and downloads the city-pair file once into data/raw/github/ if it is missing)
# and WRITES only outputs/dashboard/.
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
import calendar, json, html as _html
from pathlib import Path
import numpy as np, pandas as pd, requests
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from plotly.colors import sample_colorscale
from plotly.offline import get_plotlyjs
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist, squareform
from IPython.display import display, Markdown

SHOW_FIGURES = True        # False = only build the HTML file (faster, smaller notebook)
TODAY = pd.Timestamp.today().date().isoformat()
_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents] if (p / "data" / "clean").is_dir())
RAW, INTERIM, CLEAN = (PROJECT_ROOT / "data" / d for d in ("raw", "interim", "clean"))
OUT = PROJECT_ROOT / "outputs" / "dashboard"; OUT.mkdir(parents=True, exist_ok=True)
latest = lambda folder, pat: sorted(folder.glob(pat))[-1]

clean = pd.read_parquet(latest(CLEAN, "dgca_domestic_clean_*.parquet"))                    # domestic, cleaned
interim = pd.read_parquet(latest(INTERIM, "dgca_domestic_airline_monthly_*.parquet"))      # includes the international sections
rec = pd.read_csv(latest(CLEAN, "total_domestic_reconciliation_*.csv"), parse_dates=["Date"])
LAST = rec.loc[rec.within_2pct, "Date"].max()          # last month where the airlines add up to DGCA's own total
MIN_PAX = 25_000            # an airline is scored only if it averaged at least 25,000 passengers a month over the window
MIN_BASE_SHARE = 0.005      # 2019 comparisons need at least 0.5% of 2019 passengers
MATURE = 36                 # months an airline needs before its momentum scores count in full (countermeasure for new airlines)
print(f"clean rows: {len(clean):,} | analysis ends: {LAST:%B %Y}")


clean rows: 1,309 | analysis ends: August 2026


## 2. Monthly matrices per airline group
Domestic from the clean table, international from the interim table (same DGCA files). Months with no report inside an airline's active period are structural zeros: April-May 2020 for domestic, April 2020 to early 2022 for scheduled international flights.

In [2]:
# GROUP-MONTH MATRICES (one column per airline group): domestic from the clean table, international from the interim table
months = pd.date_range("2017-01-01", LAST, freq="MS")
FIELDS = dict(pax="Passenger Number", rpk="Passenger Kilometers", ask="Seat Kilometers", dep="Aircraft Number", hrs="Aircraft Hours")
# units: pax = passengers, rpk/ask = THOUSAND passenger-km / seat-km, dep = departures, hrs = aircraft hours

def matrices(df):
    out = {k: df.groupby(["Group", "Date"])[c].sum(min_count=1).unstack("Group").reindex(months).astype(float) for k, c in FIELDS.items()}
    return out

def zero_fill(Pm, start, end):
    """Months with no report between an airline's first and last report are a structural zero (nobody flew), not unknown values."""
    for k in Pm:
        for g in Pm[k].columns:
            s = Pm["pax"][g]; f, l = s.first_valid_index(), s.last_valid_index()
            if f is None: continue
            gap = (Pm[k].index >= max(f, pd.Timestamp(start))) & (Pm[k].index <= min(l, pd.Timestamp(end))) & Pm[k][g].isna()
            Pm[k].loc[gap, g] = 0.0
    return Pm

sd = clean[(clean.Type == "ScheduledDomestic") & (clean.Date <= LAST)]
P = zero_fill(matrices(sd), "2020-04-01", "2020-05-01")        # domestic: nobody flew in April-May 2020
mk = P["pax"].sum(axis=1); mk_rpk = P["rpk"].sum(axis=1)      # whole market

# international sections of the same DGCA files (interim table), brands mapped to the same groups as the clean table
GMAP = dict(clean[["Airline", "Group"]].drop_duplicates().values)
it = interim[(interim.Type == "ScheduledInternational") & ~interim.Airline.isin(["Total Domestic", "Total International"])].copy()
it["Date"] = pd.to_datetime(dict(year=it.Year, month=it.Month, day=1)); it = it[it.Date <= LAST]
it["Group"] = it.Airline.map(GMAP).fillna(it.Airline)
PI = zero_fill(matrices(it.dropna(subset=["Passenger Number"])), "2020-04-01", "2022-03-01")   # scheduled international flights were suspended in this period

b19i = months[months.year == 2019]
b19 = {k: P[k].loc[b19i] for k in P}
b19m = {k: b19[k].set_axis(b19i.month) for k in P}            # 2019 values indexed by month number
share19 = b19["pax"].sum() / b19["pax"].sum().sum()
BASE = [g for g in P["pax"].columns if (b19["pax"][g] > 0).sum() == 12 and share19[g] >= MIN_BASE_SHARE]   # groups with a fair 2019 yardstick
same19 = lambda k, g: pd.Series(months.month, index=months).map(b19m[k][g])
RATIO = pd.DataFrame({g: P["pax"][g] / same19("pax", g) for g in BASE})            # passengers vs the same month of 2019
RATIO_RPK = pd.DataFrame({g: P["rpk"][g] / same19("rpk", g) for g in BASE})        # passenger-km vs the same month of 2019
ROLL = RATIO.rolling(3).mean()
FIRST = P["pax"].apply(lambda s: s[s > 0].index.min())          # first month with traffic (2017-01 = in the data from the start)
age_at = lambda g, T: (T.year - FIRST[g].year) * 12 + T.month - FIRST[g].month + 1

def status_of(g):   # the project question is about passenger numbers, so the recovery status uses passengers
    tr = P["pax"][g][P["pax"][g] > 0]
    if tr.empty: return "No traffic"
    operating = tr.index.max() == LAST
    if g in BASE:
        hit = ROLL.loc["2020-07-01":, g].dropna(); hit = hit[hit >= 1]
        if len(hit): return "Recovered and holding" if (operating and RATIO[g].iloc[-3:].mean() >= 1) else "Recovered, then fell back"
        return "Never recovered (still flying)" if operating else "Never recovered (stopped flying)"
    if b19["pax"][g].sum() == 0: return "New entrant (no 2019 level)"
    return "Not tested (partial or tiny 2019 base)"
STATUS = {g: status_of(g) for g in P["pax"].columns}
print("groups with a fair 2019 yardstick:", BASE)
print(pd.Series(STATUS)[lambda s: s != "No traffic"].value_counts().to_string())


groups with a fair 2019 yardstick: ['Air India Group', 'Alliance Air', 'Go First', 'IndiGo', 'SpiceJet', 'TruJet']
New entrant (no 2019 level)               10
Not tested (partial or tiny 2019 base)     4
Recovered and holding                      2
Never recovered (stopped flying)           2
Recovered, then fell back                  1
Never recovered (still flying)             1


## 3. The ten metrics, the scoring and the countermeasures for new airlines
| Pillar | Metrics |
|---|---|
| Recovery (vs 2019) | demand recovery (passenger-km), capacity recovery (seat-km), load-factor change, consistency |
| Operations | load factor, passengers per departure, network productivity |
| Momentum | traffic growth (log), share momentum, volatility |

**Scoring:** z = (value - median) / max(1.4826 x MAD, minimum spread), capped at +/-2; score = 50 + 25 z. Pillar = mean of its scores; index = mean of the three pillars, with 50 for a missing pillar. `naive` reproduces the old method (mean/SD, % growth, no damping) so the effect of the countermeasures can be shown.

In [3]:
# THE 10 METRICS, measured "as of" a month T over the 12 months ending T (W) and the 12 months before (PW)
MET = {   # key: (label, unit, higher is better?, pillar)
    "dem":  ("Demand recovery",            "% of 2019 passenger-km",                    True,  "Recovery"),
    "cap":  ("Capacity recovery",          "% of 2019 seat-km",                         True,  "Recovery"),
    "lfd":  ("Load-factor change vs 2019", "percentage points",                         True,  "Recovery"),
    "con":  ("Consistency vs 2019",        "% of months with passenger-km at or above 2019", True, "Recovery"),
    "lf":   ("Load factor",                "% of seat-km filled",                       True,  "Operations"),
    "ppd":  ("Passengers per departure",   "passengers",                                True,  "Operations"),
    "prod": ("Network productivity",       "thousand passenger-km per aircraft hour",   True,  "Operations"),
    "gro":  ("Traffic growth",             "log change in passenger-km vs previous 12 months (x100)", True, "Momentum"),
    "shm":  ("Share momentum",             "pp change in share of passenger-km",        True,  "Momentum"),
    "vol":  ("Volatility",                 "std. dev. of monthly log year-on-year change (x100)", False, "Momentum"),
}
FORMULA = {
    "dem":  ("Sum of Passenger Kilometers, last 12 months / same sum for 2019 x 100", "Passenger Kilometers"),
    "cap":  ("Sum of Seat Kilometers, last 12 months / same sum for 2019 x 100", "Seat Kilometers"),
    "lfd":  ("Load factor, last 12 months - load factor, 2019 (both = sum of passenger-km / sum of seat-km x 100)", "Passenger Kilometers, Seat Kilometers"),
    "con":  ("Share of the last 12 months whose passenger-km is at or above the same calendar month of 2019", "Passenger Kilometers"),
    "lf":   ("Sum of Passenger Kilometers / sum of Seat Kilometers x 100, last 12 months", "Passenger Kilometers, Seat Kilometers"),
    "ppd":  ("Sum of Passenger Number / sum of Aircraft Number (departures), last 12 months", "Passenger Number, Aircraft Number"),
    "prod": ("Sum of Passenger Kilometers (thousands) / sum of Aircraft Hours, last 12 months", "Passenger Kilometers, Aircraft Hours"),
    "gro":  ("ln(passenger-km, last 12 months / passenger-km, previous 12 months) x 100", "Passenger Kilometers"),
    "shm":  ("Share of market passenger-km, last 12 months - same share, previous 12 months", "Passenger Kilometers (all airlines)"),
    "vol":  ("Standard deviation of ln(passenger-km this month / same month a year earlier) x 100, using only months whose year-earlier month had >= 25,000 passengers", "Passenger Kilometers, Passenger Number"),
}
WHY = {
    "dem": "Distance-weighted demand against the pre-COVID yardstick: a passenger flying 2,000 km counts more than one flying 300 km.",
    "cap": "Seat-km is what the airline chose to supply. Rising capacity shows confidence and funding; a cut shows retreat.",
    "lfd": "Whether planes fill as well as in 2019. Capacity can come back faster than demand, which shows up here first.",
    "con": "Separates a steady recovery from a one-month spike.",
    "lf":  "Seats filled. Low values spread fixed costs over fewer paying passengers.",
    "ppd": "Head-count per flight: aircraft size times fullness. A proxy for what each departure earns.",
    "prod": "How hard the aircraft work: passenger-km produced per hour flown.",
    "gro": "Where traffic is heading. A log change is symmetric and does not explode for a small base.",
    "shm": "Customers won from or lost to rivals, measured on passenger-km.",
    "vol": "Erratic traffic means weaker planning and revenue. Lower is better.",
}
PILLARS = ["Recovery", "Operations", "Momentum"]
MOM = [k for k in MET if MET[k][3] == "Momentum"]
FLOOR = {"dem": 10, "cap": 10, "lfd": 1.5, "con": 15, "lf": 2.5, "ppd": 15, "prod": 8, "gro": 5, "shm": 1.0, "vol": 3}   # smallest spread treated as meaningful, in the metric's units
lf_of = lambda rpk, ask: rpk / ask * 100

def metrics_asof(T, method="adjusted"):
    """method='adjusted': log growth, launch months excluded from volatility. method='naive': % growth, every month counted."""
    T = pd.Timestamp(T); i = months.get_loc(T)
    if i < 11: return pd.DataFrame(columns=list(MET))
    W = months[i - 11:i + 1]; PW = months[i - 23:i - 11] if i >= 23 else None
    rows = {}
    for g in P["pax"].columns:
        pw = P["pax"].loc[W, g]
        if not ((pw > 0).sum() == 12 and pw.mean() >= MIN_PAX): continue
        s = lambda k, ws=W: P[k].loc[ws, g].sum()
        r = dict(lf=lf_of(s("rpk"), s("ask")), ppd=s("pax") / s("dep"), prod=s("rpk") / s("hrs"))
        if PW is not None and (P["pax"].loc[PW, g] > 0).sum() == 12:
            ratio = s("rpk") / s("rpk", PW)
            r["gro"] = np.log(ratio) * 100 if method == "adjusted" else (ratio - 1) * 100
            r["shm"] = (s("rpk") / mk_rpk.loc[W].sum() - s("rpk", PW) / mk_rpk.loc[PW].sum()) * 100
            now, before = P["rpk"].loc[W, g].values, P["rpk"].loc[PW, g].values
            ok = (P["pax"].loc[PW, g].values >= MIN_PAX) if method == "adjusted" else np.ones(12, bool)
            ok &= before > 0
            if ok.sum() >= 6:
                ch = np.log(now[ok] / before[ok]) * 100 if method == "adjusted" else (now[ok] / before[ok] - 1) * 100
                r["vol"] = np.std(ch)
        if g in BASE:
            r["dem"] = s("rpk") / b19["rpk"][g].sum() * 100
            r["cap"] = s("ask") / b19["ask"][g].sum() * 100
            r["lfd"] = r["lf"] - lf_of(b19["rpk"][g].sum(), b19["ask"][g].sum())
            r["con"] = np.mean(P["rpk"].loc[W, g].values >= b19m["rpk"][g].reindex(W.month).values) * 100
        rows[g] = r
    return pd.DataFrame(rows).T.reindex(columns=list(MET)).astype(float)

def score(M, T, method="adjusted"):
    """adjusted: centre = median, spread = max(1.4826 x MAD, FLOOR); momentum pulled toward 50 for airlines younger than MATURE months;
    a missing pillar counts as 50 (no evidence = average). naive: mean and standard deviation, no damping, average of available pillars."""
    T = pd.Timestamp(T); S = pd.DataFrame(np.nan, index=M.index, columns=M.columns)
    for k in M.columns:
        x = M[k].dropna()
        if len(x) < 3: continue
        if method == "adjusted":
            c = x.median(); sc = max(1.4826 * (x - c).abs().median(), FLOOR[k])
        else:
            c = x.mean(); sc = x.std(ddof=0) or 1.0
        z = ((x - c) / sc).clip(-2, 2)
        S.loc[x.index, k] = 50 + 25 * (z if MET[k][2] else -z)
    if method == "adjusted":
        for g in S.index:
            w = min(1.0, age_at(g, T) / MATURE)
            S.loc[g, MOM] = 50 + (S.loc[g, MOM] - 50) * w
    PIL = pd.DataFrame({p: S[[k for k in MET if MET[k][3] == p]].mean(axis=1) for p in PILLARS})
    PIL["Index"] = PIL[PILLARS].fillna(50).mean(axis=1) if method == "adjusted" else PIL[PILLARS].mean(axis=1)
    PIL["Pillars measured"] = PIL[PILLARS].notna().sum(axis=1)
    return S, PIL

def reweigh(PIL, w):
    w = pd.Series(w); return (PIL[PILLARS].fillna(50) * w).sum(axis=1) / w.sum()

M0 = metrics_asof(LAST); S0, PIL0 = score(M0, LAST)
MN = metrics_asof(LAST, "naive"); SN, PILN = score(MN, LAST, "naive")      # the old method, for comparison
ORDER = PIL0.sort_values("Index", ascending=False).index.tolist()
HIST = {T: (lambda m: (m,) + score(m, T))(metrics_asof(T)) for T in months[months >= "2023-06-01"]}
RKW = pd.DataFrame({n: reweigh(PIL0, w).rank(ascending=False) for n, w in {"equal": dict(Recovery=1, Operations=1, Momentum=1),
      "recovery x2": dict(Recovery=2, Operations=1, Momentum=1), "operations x2": dict(Recovery=1, Operations=2, Momentum=1),
      "momentum x2": dict(Recovery=1, Operations=1, Momentum=2)}.items()})
print(f"airlines scored at {LAST:%b %Y}:", ORDER)
display(pd.concat([PIL0.loc[ORDER, PILLARS + ["Index"]].round(1), PILN.loc[ORDER, ["Index"]].round(1).rename(columns={"Index": "Index (old method)"})], axis=1))


airlines scored at Aug 2026: ['IndiGo', 'Akasa Air', 'Air India Group', 'Fly91', 'SpiceJet', 'Star Air', 'Alliance Air']


,Recovery,Operations,Momentum,Index,Index (old method)
IndiGo,65.5,57.2,56.4,59.7,67.7
Akasa Air,NaN,68.2,54.2,57.5,71.0
Air India Group,69.7,50.9,36.9,52.5,58.5
Fly91,NaN,13.2,70.5,44.6,41.8
SpiceJet,27.4,57.2,38.0,40.9,46.6
Star Air,NaN,12.4,59.3,40.6,43.1
Alliance Air,37.9,0.9,15.2,18.0,23.9


## 4. Does the index predict anything?
Rank correlation of each measure with the next six months of passenger-km growth, a backtest in Go First's final month, and the error of a simple projection rule.

In [4]:
# PREDICTION TESTS: each uses only information available at the time, then looks forward
spearman = lambda a, b: a.rank().corr(b.rank())
rows = []
for T, (Mt, St, Pt) in HIST.items():
    i = months.get_loc(T)
    if i + 6 >= len(months) or len(Mt) < 5: continue
    fut, prev = months[i + 1:i + 7], months[i - 11:i - 5]
    for g in Mt.index:   # outcome: passenger-km in the next 6 months vs the same 6 months a year earlier (seasonality cancels)
        y = np.log(P["rpk"].loc[fut, g].sum() / P["rpk"].loc[prev, g].sum()) * 100
        rows.append(dict(T=T, g=g, y=y, Index=Pt.loc[g, "Index"], **{k: Mt.loc[g, k] for k in MET}, **{p: Pt.loc[g, p] for p in PILLARS}))
OBS = pd.DataFrame(rows)
PRED = pd.Series({c: spearman(OBS[[c, "y"]].dropna()[c], OBS[[c, "y"]].dropna()["y"]) for c in list(MET) + PILLARS + ["Index"]})
for k in MET:                      # sign so that + means "the metric's better direction goes with faster growth"
    if not MET[k][2]: PRED[k] = -PRED[k]
NOBS = OBS[list(MET) + PILLARS + ["Index"]].notna().sum()
print(f"{len(OBS)} airline-months, {OBS['T'].min():%b %Y} to {OBS['T'].max():%b %Y}. Rank correlation with the next 6 months of passenger-km growth:")
print(PRED.round(2).sort_values(ascending=False).to_string())

BT_T = pd.Timestamp("2023-05-01")       # Go First's last month of flying
BT_M = metrics_asof(BT_T); BT_S, BT_P = score(BT_M, BT_T)
BT_rank = BT_P["Index"].rank(ascending=False).astype(int)
print(f"\nBacktest as of {BT_T:%b %Y}: Go First ranks {BT_rank['Go First']} of {len(BT_P)} (index {BT_P.loc['Go First', 'Index']:.1f})")

def forecast(s, T, h=6):    # same month last year x (last 12 months / the 12 months before)
    i = s.index.get_loc(T); g = s.iloc[i - 11:i + 1].sum() / s.iloc[i - 23:i - 11].sum()
    return pd.Series([s.iloc[i + k - 12] * g for k in range(1, h + 1)])
errs = {h: [] for h in range(1, 7)}
for T in months[(months >= "2024-08-01") & (months <= months[-7])]:
    f = forecast(mk_rpk, T)
    for h in range(1, 7): errs[h].append(abs(f.iloc[h - 1] / mk_rpk.iloc[months.get_loc(T) + h] - 1))
MAPE = pd.Series({h: np.mean(e) * 100 for h, e in errs.items()})
print("Projection test (passenger-km), mean absolute % error by months ahead:", MAPE.round(1).to_dict())


190 airline-months, Jul 2023 to Feb 2026. Rank correlation with the next 6 months of passenger-km growth:
gro           0.75
con           0.61
Recovery      0.52
shm           0.50
Momentum      0.49
cap           0.41
dem           0.39
Index         0.38
ppd           0.27
vol           0.12
prod          0.01
Operations    0.01
lf           -0.02
lfd          -0.38

Backtest as of May 2023: Go First ranks 5 of 5 (index 34.7)
Projection test (passenger-km), mean absolute % error by months ahead: {1: 4.6, 2: 5.1, 3: 5.7, 4: 5.7, 5: 6.1, 6: 6.4}


## 5. City-pair data: load and clean
Second, independent source. Cleaning steps are logged with before/after counts: city-name standardisation (194 to 144 names), merging A-to-B with B-to-A, and adding split records. Metros: Delhi, Mumbai, Bengaluru, Hyderabad, Chennai, Kolkata.

In [5]:
# CITY-PAIR DATA (second source): load, clean, log every step with before/after counts
gh_dir = RAW / "github"; gh_dir.mkdir(parents=True, exist_ok=True)
files = sorted(gh_dir.glob("vonter_domestic_city_*.csv"))
if files:
    CITY_FILE = files[-1]
else:   # download once, from the same GitHub commit the collection notebook recorded, and keep it as a raw file
    meta = json.load(open(latest(INTERIM, "collection_metadata_*.json"))); sha = meta.get("github_commit", "main")
    url = f"https://raw.githubusercontent.com/Vonter/india-aviation-traffic/{sha}/aggregated/domestic/city.csv"
    r = requests.get(url, headers={"User-Agent": "DA2402 data curation course project"}, timeout=60); r.raise_for_status()
    CITY_FILE = gh_dir / f"vonter_domestic_city_{TODAY}.csv"; CITY_FILE.write_bytes(r.content)
cp = pd.read_csv(CITY_FILE)
CLOG = [("raw file " + CITY_FILE.name, len(cp), len(cp), f"{cp.shape[1]} columns; Year, Month, City1, City2, PaxToCity2, PaxFromCity2, freight and mail")]
print("raw shape:", cp.shape); print(cp.dtypes.to_string())

CITY_MAP = {"ALLAHABAD": "PRAYAGRAJ", "ALLAHABAD (PRAYAGRAJ)": "PRAYAGRAJ", "BHATINDA": "BATHINDA", "COCHIN": "KOCHI", "CUDDAPAH": "KADAPA",
    "CUDDAPAH (KADAPA)": "KADAPA", "DEHRA DUN": "DEHRADUN", "DABOLIM": "GOA", "GOA (DABOLIM, SOUTH GOA)": "GOA", "GOA (MOPA, NORTH GOA)": "GOA",
    "MOPA, GOA": "GOA", "GULBARGA": "KALABURAGI", "HISSAR": "HISAR", "HOLLONGI, ITANAGAR": "ITANAGAR", "HIRASAR (RAJKOT)": "RAJKOT",
    "MANGALORE (MANGALURU)": "MANGALORE", "MUMBAI (MUMBAI)": "MUMBAI", "MUMBAI (NAVI MUMBAI)": "MUMBAI", "NASIK": "NASHIK",
    "PONDICHERRY": "PUDUCHERRY", "PONDICHERRY (PUDUCHERRY)": "PUDUCHERRY", "PURNEA": "PURNIA", "SIMLA": "SHIMLA", "SHOLAPUR": "SOLAPUR",
    "TIRUCHIRAPALLY": "TIRUCHIRAPPALLI", "TRIVANDRUM": "THIRUVANANTHAPURAM", "VIJAYWADA": "VIJAYAWADA",
    "VISHAKHAPATNAM (VISAKHAPATNAM)": "VISAKHAPATNAM", "ZERO": "ZIRO", "HINDON": "GHAZIABAD"}
import re
def canon(x):   # rule: trim spaces (incl. non-breaking), upper-case, drop 'AIRPORT' / 'INTERNATIONAL AIRPORT' / ', KARNATAKA', then map old and alternative names
    x = re.sub(r"\s+", " ", str(x).replace("\xa0", " ")).strip().upper()
    x = re.sub(r",\s*KARNATAKA$", "", x); x = re.sub(r"\s*(INTERNATIONAL )?AIRPORT$", "", x).strip()
    return CITY_MAP.get(x, x)
n_names = len(set(cp.City1) | set(cp.City2))
cp["A"], cp["B"] = cp.City1.map(canon), cp.City2.map(canon)
CLOG.append(("standardise city names (spaces, case, 'AIRPORT', renamed cities, two Goa airports, Navi Mumbai)", n_names, len(set(cp.A) | set(cp.B)), "unit = distinct city names"))
n = len(cp); cp = cp[cp.A != cp.B]; CLOG.append(("drop rows whose two ends became the same city", n, len(cp), "rows"))
cp["pax"] = cp.PaxToCity2 + cp.PaxFromCity2                               # both directions = passengers on the route
cp["Date"] = pd.to_datetime(dict(year=cp.Year, month=cp.Month, day=1))
cp["p1"], cp["p2"] = np.where(cp.A < cp.B, cp.A, cp.B), np.where(cp.A < cp.B, cp.B, cp.A)
dup = int(cp.duplicated(["Date", "City1", "City2"]).sum())
n_dir = len(set(zip(cp.City1, cp.City2))); n_pairs = len(set(zip(cp.p1, cp.p2)))
CR = cp.groupby(["Date", "p1", "p2"], as_index=False).pax.sum()             # one row per route per month
CLOG.append((f"merge A-to-B and B-to-A, renamed-city rows and {dup} split records (same route and month) into one route", len(cp), len(CR), f"rows; {n_dir:,} directional pairs -> {n_pairs:,} routes"))
LASTC = min(CR.Date.max(), LAST); CR = CR[CR.Date <= LASTC]
WC = pd.date_range(LASTC - pd.DateOffset(months=11), LASTC, freq="MS")
METRO = ["DELHI", "MUMBAI", "BENGALURU", "HYDERABAD", "CHENNAI", "KOLKATA"]
CR["kind"] = np.select([CR.p1.isin(METRO) & CR.p2.isin(METRO), CR.p1.isin(METRO) | CR.p2.isin(METRO)], ["Metro to metro", "Metro to other city"], "Other city to other city")
ctot = CR.groupby("Date").pax.sum()
XCHK = (ctot / mk.reindex(ctot.index)).dropna()
XCHK = XCHK[XCHK.index >= "2017-01-01"]
display(pd.DataFrame(CLOG, columns=["step", "before", "after", "note"]))
print(f"city-pair data ends {LASTC:%B %Y}; city-pair total / airline total: median {XCHK.median():.3f}, range {XCHK.min():.3f} to {XCHK.max():.3f} over {len(XCHK)} months")


raw shape: (66453, 10)
Year                  int64
Month                 int64
City1                   str
City2                   str
PaxToCity2          float64
PaxFromCity2        float64
FreightToCity2      float64
FreightFromCity2    float64
MailToCity2         float64
MailFromCity2       float64


,step,before,after,note
0,raw file vonter_domestic_city_2026-11-01.csv,66453,66453,"10 columns; Year, Month, City1, City2, PaxToCi..."
1,"standardise city names (spaces, case, 'AIRPORT...",194,144,unit = distinct city names
2,drop rows whose two ends became the same city,66453,66448,rows
3,"merge A-to-B and B-to-A, renamed-city rows and...",66448,65764,"rows; 2,183 directional pairs -> 1,470 routes"


city-pair data ends July 2026; city-pair total / airline total: median 1.000, range 0.994 to 1.046 over 112 months


## 6. Chart registry, colours and the worked example

In [6]:
# CHART REGISTRY: each chart is stored with its section, a plain title, a "how to read" line and a caption whose numbers are computed
FIGS = {}
SEC = {1: "The ten metrics: what each one measures", 2: "The scorecard: who performs best?", 3: "Recovery: who regained 2019?",
       4: "Operations: how well are seats and aircraft used?", 5: "Momentum and market share: where is each airline heading?",
       6: "The route network: city-pair data", 7: "International flights", 8: "Relationships and prediction"}
# Airline colours: checked with a colour-blind palette validator. Colour follows the airline, never its rank; symbols are a second cue.
COL = {"IndiGo": "#0072B2", "Air India Group": "#D55E00", "SpiceJet": "#E69F00", "Akasa Air": "#009E73", "Star Air": "#56B4E9",
       "Alliance Air": "#CC79A7", "Fly91": "#A0522D", "Go First": "#6A3D9A", "TruJet": "#8E44AD", "Jet Airways": "#333333"}
SYM = {"IndiGo": "circle", "Air India Group": "square", "SpiceJet": "diamond", "Akasa Air": "triangle-up", "Star Air": "x",
       "Alliance Air": "star", "Fly91": "hexagon", "Go First": "cross", "TruJet": "pentagon", "Jet Airways": "triangle-down"}
gc = lambda g: COL.get(g, "#999999"); gs = lambda g: SYM.get(g, "circle")
VIR = lambda v: sample_colorscale("Viridis", [float(np.clip(v, 0, 100)) / 100])[0]
DIV = [[0, "#B35806"], [0.5, "#F2F2F2"], [1, "#2166AC"]]          # diverging: orange below, grey neutral, blue above
f1 = lambda v: f"{v:,.1f}"; fmt = lambda v: "" if pd.isna(v) else (f"{v:.1f}" if abs(v) < 10 else f"{v:.0f}")
bn = lambda v: v / 1e6                                             # thousand km -> billion km
def put(key, sec, title, how, fig, cap, h=470):
    m = fig.layout.margin
    fig.update_layout(template="plotly_white", font=dict(family="system-ui, 'Segoe UI', Arial, sans-serif", size=13, color="#222"), height=h,
                      margin=dict(l=60 if m.l is None else m.l, r=30 if m.r is None else m.r, t=30 if m.t is None else m.t, b=55 if m.b is None else m.b),
                      legend=dict(orientation="h", y=-0.2, x=0))
    FIGS[key] = dict(sec=sec, title=title, how=how, fig=fig, cap=cap)
COMMON = ["lf", "ppd", "prod", "gro", "shm", "vol"]          # metrics every scored airline can have (no 2019 yardstick needed)
top, bot = ORDER[0], ORDER[-1]
W12 = months[-12:]
FULL = PIL0[PIL0["Pillars measured"] == 3].sort_values("Index", ascending=False)

# worked example: every metric for the top airline, with the numbers that go into it
def worked(g, T=LAST):
    i = months.get_loc(T); W, PW = months[i - 11:i + 1], months[i - 23:i - 11]
    s = lambda k, ws=W: P[k].loc[ws, g].sum()
    v = M0.loc[g]; out = {}
    if g in BASE:
        out["dem"] = f"{bn(s('rpk')):.1f} bn / {bn(b19['rpk'][g].sum()):.1f} bn x 100 = {v['dem']:.1f}%"
        out["cap"] = f"{bn(s('ask')):.1f} bn / {bn(b19['ask'][g].sum()):.1f} bn x 100 = {v['cap']:.1f}%"
        out["lfd"] = f"{v['lf']:.2f} - {lf_of(b19['rpk'][g].sum(), b19['ask'][g].sum()):.2f} = {v['lfd']:+.2f} pp"
        out["con"] = f"{int(round(v['con'] * 12 / 100))} of 12 months = {v['con']:.0f}%"
    out["lf"] = f"{bn(s('rpk')):.1f} bn / {bn(s('ask')):.1f} bn x 100 = {v['lf']:.1f}%"
    out["ppd"] = f"{s('pax'):,.0f} / {s('dep'):,.0f} = {v['ppd']:.1f}"
    out["prod"] = f"{s('rpk'):,.0f} / {s('hrs'):,.0f} = {v['prod']:.1f}"
    out["gro"] = f"ln({bn(s('rpk')):.1f} / {bn(s('rpk', PW)):.1f}) x 100 = {v['gro']:+.1f}"
    out["shm"] = f"{s('rpk') / mk_rpk.loc[W].sum() * 100:.2f}% - {s('rpk', PW) / mk_rpk.loc[PW].sum() * 100:.2f}% = {v['shm']:+.2f} pp"
    out["vol"] = f"spread of 12 monthly changes = {v['vol']:.1f}"
    return out
WORKED = worked("IndiGo")


## 7. Sections 1 and 2: the metrics and the scorecard

In [7]:
# ---------------- SECTION 1: THE TEN METRICS ----------------
# 1.1 SMALL-MULTIPLE DOT PLOT: the raw value of every metric for every airline, against the peer median
SHORTU = {"dem": "% of 2019 pax-km", "cap": "% of 2019 seat-km", "lfd": "points vs 2019", "con": "% of months >= 2019", "lf": "% of seat-km filled",
          "ppd": "passengers", "prod": "000 pax-km per hour", "gro": "log points", "shm": "points of share", "vol": "log points"}
seen = set()
fig = make_subplots(rows=2, cols=5, subplot_titles=[f"{MET[k][0]}<br><sup>{SHORTU[k]}</sup>" for k in MET], shared_yaxes=True,
                    horizontal_spacing=0.03, vertical_spacing=0.2)
for j, k in enumerate(MET):
    r, c = j // 5 + 1, j % 5 + 1; x = M0[k].dropna()
    for g in ORDER:
        if g in x.index:
            fig.add_trace(go.Scatter(x=[x[g]], y=[g], mode="markers", marker=dict(color=gc(g), symbol=gs(g), size=11, line=dict(color="white", width=1)),
                                     name=g, legendgroup=g, showlegend=(g not in seen), hovertemplate=f"{g}<br>{MET[k][0]}: %{{x:.1f}}<extra></extra>"), row=r, col=c)
            seen.add(g)
    if len(x): fig.add_vline(x=float(x.median()), line=dict(dash="dot", color="#888", width=1), row=r, col=c)
fig.update_yaxes(categoryorder="array", categoryarray=ORDER[::-1], tickfont=dict(size=11)); fig.update_xaxes(tickfont=dict(size=10)); fig.update_annotations(font_size=11)
fig.update_layout(margin=dict(l=120, r=20, t=60, b=40), legend=dict(orientation="h", y=-0.12))
put("m_dots", 1, "What each metric shows, airline by airline", "One panel per metric, in the units listed in the table above. Each symbol is an airline; the dotted line is the median of the scored airlines. Recovery panels only show airlines with a full 2019 year.", fig,
    f"IndiGo carries {M0.loc['IndiGo', 'dem']:.0f}% of its 2019 passenger-km, Air India Group {M0.loc['Air India Group', 'dem']:.0f}% and SpiceJet {M0.loc['SpiceJet', 'dem']:.0f}%. "
    f"{M0['lf'].idxmax()} fills the most seats ({M0['lf'].max():.0f}%) and {M0['gro'].idxmax()} grows fastest ({M0['gro'].max():+.0f} log points); "
    f"turboprop operators sit at the bottom of the operations panels ({M0['ppd'].idxmin()}: {M0['ppd'].min():.0f} passengers per departure).", 640)

# 1.2 DUMBBELL: what the countermeasures for new airlines change
cmp = pd.DataFrame({"old": PILN["Index"], "new": PIL0["Index"]}).loc[ORDER]
fig = go.Figure()
for g in ORDER:
    fig.add_trace(go.Scatter(x=[cmp.loc[g, "old"], cmp.loc[g, "new"]], y=[g, g], mode="lines", line=dict(color="#c8c8c8", width=5), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=cmp.old, y=ORDER, mode="markers", name="Old method (mean/SD, % growth, no damping)", marker=dict(symbol="circle-open", size=13, color="#555", line=dict(width=2.5))))
fig.add_trace(go.Scatter(x=cmp.new, y=ORDER, mode="markers", name="Adjusted method (used everywhere on this page)", marker=dict(symbol="diamond", size=13, color="#0072B2")))
fig.add_trace(go.Scatter(x=cmp.max(axis=1) + 2, y=ORDER, mode="text", text=[f"{d:+.1f}" for d in cmp.new - cmp.old], textposition="middle right", textfont=dict(size=11), showlegend=False, hoverinfo="skip"))
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title="Performance index (0 to 100)", range=[0, 90])
fig.update_layout(margin=dict(l=130))
chg = (cmp.new - cmp.old).sort_values()
put("m_counter", 1, "What the countermeasures for new airlines change", "Each line joins an airline's index under the old method (circle) and the adjusted method (diamond). The number is the change.", fig,
    f"{chg.index[0]} changes most ({chg.iloc[0]:+.1f} points: it no longer tops the table on the strength of two pillars), and {chg.index[-1]} moves {chg.iloc[-1]:+.1f}. "
    f"Under the old method the leader was {PILN['Index'].idxmax()} ({PILN['Index'].max():.0f}); under the adjusted method it is {top} ({PIL0.loc[top, 'Index']:.0f}).", 430)

# ---------------- SECTION 2: THE SCORECARD ----------------
PILp = HIST[LAST - pd.DateOffset(months=12)][2]
fig = make_subplots(rows=2, cols=4, specs=[[{"type": "indicator"}] * 4] * 2, vertical_spacing=0.38)
fig.update_layout(margin=dict(t=70, b=20))
for k, g in enumerate(ORDER):
    d = dict(reference=float(PILp.loc[g, "Index"]), valueformat=".0f") if g in PILp.index else {}
    fig.add_trace(go.Indicator(mode="gauge+number" + ("+delta" if d else ""), value=float(PIL0.loc[g, "Index"]), number=dict(valueformat=".0f"), delta=d,
        title=dict(text=g + ("" if PIL0.loc[g, "Pillars measured"] == 3 else "<br><span style='font-size:11px'>2 of 3 pillars</span>"), font=dict(size=14)),
        gauge=dict(axis=dict(range=[0, 100]), bar=dict(color=gc(g), thickness=.55), steps=[dict(range=[0, 40], color="#efefef"), dict(range=[40, 60], color="#e2e2e2"), dict(range=[60, 100], color="#d4d4d4")])),
        row=k // 4 + 1, col=k % 4 + 1)
put("gauges", 2, "How does each airline score today?", "Filled arc and number = performance index (0 to 100, 50 = typical of the scored airlines). The arrow is the change since a year ago.", fig,
    f"{top} has the highest index ({PIL0.loc[top, 'Index']:.0f}) and {bot} the lowest ({PIL0.loc[bot, 'Index']:.0f}). "
    f"Airlines that started after 2019 are marked '2 of 3 pillars': their missing Recovery pillar counts as 50, so they can neither gain nor lose from it.", 500)

z = S0.loc[ORDER]; txt = [[fmt(M0.loc[g, k]) for k in MET] for g in ORDER]
fig = go.Figure(go.Heatmap(z=z.values, x=[MET[k][0] for k in MET], y=ORDER, colorscale="Viridis", zmin=0, zmax=100, text=txt, texttemplate="%{text}", xgap=2, ygap=2,
                           colorbar=dict(title=dict(text="Score")), hovertemplate="%{y}<br>%{x}<br>score %{z:.0f}, value %{text}<extra></extra>"))
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(tickangle=-30)
best = S0.stack().idxmax(); worst = S0.stack().idxmin()
put("heatmap", 2, "Where is each airline strong or weak?", "Colour = score (yellow strong, purple weak); number = the raw value. Blank = no 2019 level to compare with.", fig,
    f"The best single score is {best[0]} on {MET[best[1]][0].lower()} ({S0.loc[best]:.0f}); the weakest is {worst[0]} on {MET[worst[1]][0].lower()} ({S0.loc[worst]:.0f}). "
    f"Read across a row to see whether an airline is a specialist or an all-rounder.", 440)

THEAD = {"dem": "Demand<br>recovery", "cap": "Capacity<br>recovery", "lfd": "LF change<br>vs 2019", "con": "Consistency<br>vs 2019", "lf": "Load<br>factor",
         "ppd": "Pax per<br>departure", "prod": "Network<br>productivity", "gro": "Traffic<br>growth", "shm": "Share<br>momentum", "vol": "Volatility"}
hdr = ["#", "Airline", "Index", "Recov-<br>ery", "Oper-<br>ations", "Mom-<br>entum"] + [THEAD[k] for k in MET]
cells, fills = [[i + 1 for i in range(len(ORDER))], ORDER], [["#fff"] * len(ORDER)] * 2
for c in ["Index", "Recovery", "Operations", "Momentum"]:
    cells.append([("n/a" if pd.isna(PIL0.loc[g, c]) else f"{PIL0.loc[g, c]:.0f}") for g in ORDER]); fills.append([("#f2f2f2" if pd.isna(PIL0.loc[g, c]) else VIR(PIL0.loc[g, c])) for g in ORDER])
for k in MET:
    cells.append([("n/a" if pd.isna(M0.loc[g, k]) else f1(M0.loc[g, k])) for g in ORDER]); fills.append(["#fff"] * len(ORDER))
fcol = [["#222"] * len(ORDER)] * 2 + [["white" if (not pd.isna(PIL0.loc[g, c]) and PIL0.loc[g, c] < 60) else "#222" for g in ORDER] for c in ["Index", "Recovery", "Operations", "Momentum"]] + [["#222"] * len(ORDER)] * len(MET)
fig = go.Figure(go.Table(columnwidth=[26, 110, 52, 60, 62, 62] + [82] * len(MET), header=dict(values=hdr, fill_color="#222", font=dict(color="white", size=10.5), align="center", height=46),
                         cells=dict(values=cells, fill_color=fills, font=dict(color=fcol, size=12), align="center", height=30)))
put("table", 2, "The full scorecard", "Coloured cells are scores; plain cells are raw values in the units of the metrics table. n/a = no 2019 level.", fig,
    f"{top} scores {PIL0.loc[top, 'Index']:.0f} with {M0.loc[top, 'dem']:.0f}% of its 2019 passenger-km and a load factor of {M0.loc[top, 'lf']:.0f}%. "
    f"{bot} is last ({PIL0.loc[bot, 'Index']:.0f}) with a load factor of {M0.loc[bot, 'lf']:.0f}% and traffic {M0.loc[bot, 'gro']:+.0f} log points on the previous year.", 140 + 32 * len(ORDER) + 40)

th = [MET[k][0] for k in COMMON]
fig = go.Figure()
for g in ORDER:
    r_ = [S0.loc[g, k] if pd.notna(S0.loc[g, k]) else 50 for k in COMMON]
    fig.add_trace(go.Scatterpolar(r=r_ + r_[:1], theta=th + th[:1], name=g, fill="toself", opacity=.45, line=dict(color=gc(g), width=2), marker=dict(symbol=gs(g), size=8),
                                  mode="lines+markers", visible=True if g in ORDER[:4] else "legendonly"))
fig.update_layout(polar=dict(radialaxis=dict(range=[0, 100], tickvals=[25, 50, 75])))
lead = ORDER[0]
put("radar", 2, "What shape is each airline?", "Each spoke is one metric score (outer edge = best). Click names in the legend to add or remove airlines.", fig,
    f"{lead} reaches {S0.loc[lead, COMMON].max():.0f} on {MET[S0.loc[lead, COMMON].idxmax()][0].lower()} but only {S0.loc[lead, COMMON].min():.0f} on {MET[S0.loc[lead, COMMON].idxmin()][0].lower()}. "
    f"No airline is strong on every spoke, which is why the index averages several metrics.", 500)

H = pd.DataFrame({T: v[2]["Index"] for T, v in HIST.items()}).T; RK = H.rank(axis=1, ascending=False)
fig = go.Figure()
for g in RK.columns:
    s = RK[g]
    fig.add_trace(go.Scatter(x=s.index, y=s, mode="lines+markers", name=g, line=dict(color=gc(g), width=2.5), marker=dict(size=7, symbol=gs(g)),
                             hovertemplate=g + ": rank %{y:.0f} in %{x|%b %Y}<extra></extra>"))
    lv = s.dropna()
    if len(lv) and lv.index[-1] == RK.index[-1]:
        fig.add_annotation(x=lv.index[-1], y=lv.iloc[-1], text=g, xanchor="left", xshift=8, showarrow=False, font=dict(size=11))
fig.update_yaxes(autorange="reversed", title="Rank (1 = best)", dtick=1)
fig.update_xaxes(title="Month (index measured over the 12 months ending then)", range=[RK.index[0] - pd.Timedelta(days=20), RK.index[-1] + pd.Timedelta(days=150)])
chg2 = (RK.iloc[0] - RK.iloc[-1]).dropna().sort_values(); swing = (RKW.max(axis=1) - RKW.min(axis=1))
put("bump", 2, "Who has been climbing or sliding?", "Each line is an airline's rank among the scored airlines that month. An airline joins once it has two full years of data.", fig,
    f"{chg2.index[-1]} climbed {chg2.iloc[-1]:.0f} place(s) since {RK.index[0]:%b %Y}; {chg2.index[0]} moved {chg2.iloc[0]:+.0f}. {top} ranks first in {int((RK[top] == 1).sum())} of {len(RK)} months. "
    f"The ranking is robust: doubling the weight of any one pillar moves no airline more than {int(swing.max())} places.", 450)


## 8. Sections 3 to 5: recovery, operations, momentum

In [8]:
# ---------------- SECTION 3: RECOVERY ----------------
MON = [calendar.month_abbr[m] for m in range(1, 13)]
# 3.1 STACKED AREA on one time axis (replaces the 3D landscape): passenger-km by airline group
show = ["IndiGo", "Air India Group", "SpiceJet", "Go First", "Akasa Air", "Jet Airways"]
ra = P["rpk"].fillna(0) / 1e6; other = ra.drop(columns=show).sum(axis=1)
fig = go.Figure()
for g in show[::-1] + ["Other airlines"]:
    y = other if g == "Other airlines" else ra[g]
    fig.add_trace(go.Scatter(x=ra.index, y=y, name=g, stackgroup="one", mode="lines", line=dict(width=0.5, color="white"), fillcolor=gc(g) if g in COL else "#c7c7c7",
                             hovertemplate=g + ": %{y:.2f} bn passenger-km<extra></extra>"))
base_line = pd.Series(months.month, index=months).map(b19m["rpk"].sum(axis=1)) / 1e6
fig.add_trace(go.Scatter(x=months, y=base_line, name="Same month of 2019 (all airlines)", mode="lines", line=dict(color="#111", dash="dash", width=1.5)))
fig.update_yaxes(title="Passenger-km per month (billions)"); fig.update_xaxes(title="Month")
RPK12, RPK19 = mk_rpk.loc[W12].sum(), b19["rpk"].sum().sum()
put("rpk_area", 3, "The market on one time line: passenger-km by airline", "Each band is an airline group's passenger-km that month (bands stack to the market total). The dashed line repeats the same month of 2019.", fig,
    f"Over the last 12 months the market flew {RPK12 / 1e6:.0f} billion passenger-km, {(RPK12 / RPK19 - 1) * 100:+.0f}% on 2019 (passengers: {(mk.loc[W12].sum() / b19['pax'].sum().sum() - 1) * 100:+.0f}%). "
    f"IndiGo's band grew from {b19['rpk']['IndiGo'].sum() / RPK19 * 100:.0f}% to {P['rpk'].loc[W12, 'IndiGo'].sum() / RPK12 * 100:.0f}% of passenger-km, while the Go First and Jet Airways bands vanish.", 480)

# 3.2 BOX PLOT: monthly passenger-km vs the same month of 2019
since = months >= "2023-06-01"; act_base = [g for g in ORDER if g in BASE]
fig = go.Figure([go.Box(y=(RATIO_RPK.loc[since, g] * 100).dropna(), name=g, marker=dict(color=gc(g), symbol=gs(g)), boxpoints="all", jitter=.4, pointpos=-1.7, boxmean=True) for g in act_base])
fig.add_hline(y=100, line=dict(dash="dash", color="#222"), annotation_text="2019 level", annotation_position="top right")
fig.update_yaxes(title="Passenger-km, % of the same month in 2019"); fig.update_layout(showlegend=False)
med = {g: (RATIO_RPK.loc[since, g] * 100).median() for g in act_base}; above = [g for g in act_base if (RATIO_RPK.loc[since, g] * 100).quantile(.25) > 100]
put("box", 3, "How far above or below 2019, month by month?", "Box = middle half of the months since mid-2023, solid line = median, dashed = mean, dots = single months.", fig,
    f"{max(med, key=med.get)}'s median month is {max(med.values()):.0f}% of its 2019 passenger-km; {min(med, key=med.get)}'s is {min(med.values()):.0f}%. "
    + (f"Only {' and '.join(above)} sit entirely above the 2019 line." if above else "No airline sits entirely above the 2019 line."), 460)

# 3.3 WATERFALL: change in average monthly passenger-km, 2019 to the last 12 months
a19, a12 = mk_rpk.loc[b19i].mean() / 1e6, mk_rpk.loc[W12].mean() / 1e6
dlt = (P["rpk"].loc[W12].fillna(0).mean() - P["rpk"].loc[b19i].fillna(0).mean()) / 1e6
big = dlt[dlt.abs() >= 0.05].sort_values(ascending=False); rest = dlt.sum() - big.sum()
assert abs(a19 + dlt.sum() - a12) < 1e-6
fig = go.Figure(go.Waterfall(x=["2019 average"] + big.index.tolist() + ["Others", "Last 12 months"], measure=["absolute"] + ["relative"] * (len(big) + 1) + ["total"],
        y=[a19] + big.tolist() + [rest, 0], text=[f"{a19:.1f}"] + [f"{v:+.2f}" for v in big] + [f"{rest:+.2f}", f"{a12:.1f}"], textposition="outside",
        increasing=dict(marker=dict(color="#2166AC")), decreasing=dict(marker=dict(color="#B35806")), totals=dict(marker=dict(color="#555555")), connector=dict(line=dict(color="#999"))))
fig.update_yaxes(title="Passenger-km per month (billions)", range=[0, max(a19, a12) * 1.3]); fig.update_xaxes(tickangle=-25)
gone = [g for g in big.index if big[g] < 0 and P["pax"].loc[W12, g].fillna(0).sum() == 0]     # stopped flying altogether
put("waterfall", 3, "Who added and who lost passenger-km?", "Start = the average month of 2019. Blue steps add passenger-km, orange steps lose them; the dark bar is the average month of the last 12 months.", fig,
    f"The average month grew from {a19:.1f} to {a12:.1f} billion passenger-km. {big.index[0]} added {big.iloc[0]:+.2f} billion a month; airlines that stopped flying ({', '.join(gone)}) "
    f"took {sum(big[g] for g in gone):+.2f} billion with them, and SpiceJet changed by {dlt.get('SpiceJet', 0):+.2f} billion.", 480)

# 3.4 SANKEY: 2019 brands -> groups -> outcome (passengers, as in the project question)
b = sd[sd.Year == 2019].groupby(["Airline", "Group"])["Passenger Number"].sum().reset_index(); b = b[b["Passenger Number"] > 0]
b["src"] = np.where(b["Passenger Number"] < 0.3e6, "Other small brands", b.Airline)
L = b.groupby(["src", "Group"])["Passenger Number"].sum().reset_index(); gout = L.groupby("Group")["Passenger Number"].sum()
OUTC = {"Recovered and holding": "#2166AC", "Recovered, then fell back": "#E69F00", "Never recovered (still flying)": "#B35806",
        "Never recovered (stopped flying)": "#6A3D9A", "New entrant (no 2019 level)": "#7F7F7F", "Not tested (partial or tiny 2019 base)": "#BBBBBB"}
rgba = lambda h, a: f"rgba({int(h[1:3], 16)},{int(h[3:5], 16)},{int(h[5:7], 16)},{a})"
nodes = [("b", s) for s in L.src.unique()] + [("g", g) for g in gout.index] + [("o", o) for o in sorted({STATUS[g] for g in gout.index})]
nid = {n_: i for i, n_ in enumerate(nodes)}
fig = go.Figure(go.Sankey(arrangement="snap", node=dict(label=[n_ for _, n_ in nodes], color=["#bbbbbb" if t == "b" else (gc(n_) if t == "g" else OUTC[n_]) for t, n_ in nodes], pad=16, thickness=16),
        link=dict(source=[nid[("b", r.src)] for r in L.itertuples()] + [nid[("g", g)] for g in gout.index], target=[nid[("g", r.Group)] for r in L.itertuples()] + [nid[("o", STATUS[g])] for g in gout.index],
                  value=[v / 1e6 for v in L["Passenger Number"]] + [v / 1e6 for v in gout.values], color=["rgba(150,150,150,.35)"] * len(L) + [rgba(OUTC[STATUS[g]], .45) for g in gout.index],
                  hovertemplate="%{source.label} to %{target.label}: %{value:.1f} million passengers<extra></extra>")))
fig.update_layout(margin=dict(l=20, r=20, t=20, b=20))
oc = pd.Series({o: gout[[g for g in gout.index if STATUS[g] == o]].sum() / 1e6 for o in OUTC if any(STATUS[g] == o for g in gout.index)}); T19 = gout.sum() / 1e6
put("sankey", 3, "Where did the 2019 passengers end up?", "Left = 2019 brands (width = 2019 passengers), middle = airline group, right = what happened to that group since.", fig,
    f"Of {T19:.0f} million passengers in 2019, {oc.get('Recovered and holding', 0) / T19 * 100:.0f}% flew with groups that are above 2019 today, "
    f"{oc.get('Never recovered (stopped flying)', 0) / T19 * 100:.0f}% with groups that stopped flying without recovering and {oc.get('Never recovered (still flying)', 0) / T19 * 100:.0f}% with groups still below 2019.", 540)

# 3.5 FUNNEL
st = {g: STATUS[g] for g in BASE}; n0 = len(BASE)
n1 = sum(1 for g in BASE if "still flying" in st[g] or st[g].startswith("Recovered")); n2 = sum(1 for g in BASE if st[g].startswith("Recovered")); n3 = sum(1 for g in BASE if st[g] == "Recovered and holding")
fig = go.Figure(go.Funnel(y=["Flew all of 2019 (and big enough to compare)", "Still flying today", "Reached 2019 passenger level again", "Still above 2019 level today"], x=[n0, n1, n2, n3],
        textinfo="value+percent initial", marker=dict(color=["#999999", "#56B4E9", "#E69F00", "#2166AC"])))
fig.update_layout(margin=dict(l=280, r=30, t=30, b=40))
put("funnel", 3, "How many airlines made it through?", "Each stage keeps only the airlines that passed the previous test (passenger numbers, 3-month average vs 2019).", fig,
    f"Of {n0} airlines that flew all of 2019, {n1} are still flying, {n2} got back to their 2019 passenger level and {n3} are above it now.", 400)

# ---------------- SECTION 4: OPERATIONS ----------------
LFm = lf_of(P["rpk"], P["ask"]); TRIP = P["rpk"] * 1000 / P["pax"]          # monthly load factor (%) and average trip length (km)
fig = go.Figure([go.Violin(y=LFm.loc[since, g].dropna(), name=g, line=dict(color=gc(g)), fillcolor=gc(g), opacity=.55, box_visible=True, meanline_visible=True,
                           points="all", pointpos=0, jitter=.3, marker=dict(size=3), spanmode="hard") for g in ORDER])
fig.update_yaxes(title="Monthly load factor (% of seat-km filled)"); fig.update_layout(showlegend=False)
sd_lf = LFm.loc[since, ORDER].std().sort_values()
put("violin", 4, "How steady is the load factor?", "Width = how often the airline had that load factor (months since mid-2023). Short and fat = steady; tall and thin = swings.", fig,
    f"{sd_lf.index[0]} is the steadiest (standard deviation {sd_lf.iloc[0]:.1f} points) and {sd_lf.index[-1]} the most erratic ({sd_lf.iloc[-1]:.1f}). "
    f"Median load factors run from {LFm.loc[since, ORDER].median().min():.0f}% to {LFm.loc[since, ORDER].median().max():.0f}%.", 480)

rk12 = P["rpk"].loc[W12].sum(); sh = (rk12 / rk12.sum() * 100)[ORDER].sort_values(ascending=False); lf12 = {g: lf_of(P["rpk"].loc[W12, g].sum(), P["ask"].loc[W12, g].sum()) for g in ORDER}
wd = sh / sh.sum() * 100; start = wd.cumsum() - wd
fig = go.Figure(go.Bar(x=(start + wd / 2).tolist(), y=[lf12[g] for g in wd.index], width=(wd * 0.97).tolist(), marker=dict(color=[gc(g) for g in wd.index], line=dict(color="white", width=1)),
        text=[f"{g}<br>{lf12[g]:.0f}%" if w > 8 else "" for g, w in wd.items()], textposition="inside", insidetextanchor="start", textfont=dict(color="white"),
        hovertext=[f"{g}: {w:.1f}% of passenger-km, load factor {lf12[g]:.1f}%" for g, w in wd.items()], hoverinfo="text"))
for g, w in wd.items():
    if 2.5 < w <= 8: fig.add_annotation(x=start[g] + w / 2, y=lf12[g], text=f"{g} {lf12[g]:.0f}%", textangle=-90, yanchor="bottom", showarrow=False, font=dict(size=10))
fig.update_xaxes(title="Cumulative share of passenger-km (%): bar width = the airline's share", range=[0, 100]); fig.update_yaxes(title="Load factor (%)", range=[0, 135])
put("marimekko", 4, "Big and full at the same time?", "Width = share of passenger-km over the last 12 months, height = load factor. The thinnest slivers are labelled on hover.", fig,
    f"{wd.index[0]} flies {wd.iloc[0]:.0f}% of all passenger-km at a {lf12[wd.index[0]]:.0f}% load factor; weighted by passenger-km the market fills {sum(lf12[g] * w for g, w in wd.items()) / 100:.0f}% of seat-km.", 460)

trip = {g: P["rpk"].loc[W12, g].sum() * 1000 / P["pax"].loc[W12, g].sum() for g in ORDER}
askb = np.array([bn(P["ask"].loc[W12, g].sum()) for g in ORDER])
fig = go.Figure(go.Scatter(x=[trip[g] for g in ORDER], y=M0.loc[ORDER, "ppd"], mode="markers+text", text=ORDER, textposition=["bottom center" if g == "SpiceJet" else "top center" for g in ORDER],
        marker=dict(size=np.sqrt(askb) * 5 + 10, color=[lf12[g] for g in ORDER], colorscale="Viridis", cmin=60, cmax=95, showscale=True, colorbar=dict(title=dict(text="Load factor %")),
                    line=dict(color="#222", width=1)), hovertemplate="%{text}<br>average trip %{x:.0f} km<br>%{y:.0f} passengers per departure<extra></extra>"))
fig.update_xaxes(title="Average trip length (passenger-km / passengers, km)"); fig.update_yaxes(title="Passengers per departure")
put("trip", 4, "Two business models: long trips on big jets, short hops on small planes", "Each bubble is an airline (last 12 months). Size = seat-km offered, colour = load factor.", fig,
    f"{max(trip, key=trip.get)} has the longest average trip ({max(trip.values()):.0f} km) and {min(trip, key=trip.get)} the shortest ({min(trip.values()):.0f} km). "
    f"The jet airlines carry {M0.loc[[g for g in ORDER if M0.loc[g, 'ppd'] > 100], 'ppd'].min():.0f} to {M0['ppd'].max():.0f} passengers per departure; the turboprop operators {M0['ppd'].min():.0f} to {M0.loc[[g for g in ORDER if M0.loc[g, 'ppd'] < 100], 'ppd'].max():.0f}.", 480)

dd = pd.DataFrame({"trip": TRIP.loc[since, ORDER].stack(), "lf": LFm.loc[since, ORDER].stack()}).dropna()
fig = go.Figure([go.Histogram2dContour(x=dd.trip, y=dd.lf, colorscale=[[0, "#ffffff"], [0.12, "#deebf7"], [0.5, "#4292c6"], [1, "#08306b"]], ncontours=12, showscale=True,
                                       colorbar=dict(title=dict(text="Months")), contours=dict(coloring="fill"), line=dict(width=0.5, color="#fff")),
                 go.Scatter(x=dd.trip, y=dd.lf, mode="markers", marker=dict(size=3, color="rgba(40,40,40,.55)"), hoverinfo="skip", showlegend=False)])
fig.update_xaxes(title="Average trip length (km)"); fig.update_yaxes(title="Monthly load factor (%)")
put("contour", 4, "Do longer trips fill planes better?", "Contours show where airline-months cluster (darker = more months); grey dots are single airline-months since mid-2023.", fig,
    f"The rank correlation between trip length and load factor is {spearman(dd.trip, dd.lf):+.2f} across {len(dd)} airline-months: longer-haul networks fill a larger share of their seats.", 460)

br = sd[sd.Date.isin(W12)].groupby(["Group", "Airline"])[["Passenger Kilometers"]].sum().reset_index(); br = br[(br["Passenger Kilometers"] > 0) & br.Group.isin(ORDER)]
ids, labels, parents, values, cols = ["root"], ["Scheduled domestic"], [""], [bn(br["Passenger Kilometers"].sum())], [lf_of(rk12[ORDER].sum(), P["ask"].loc[W12, ORDER].sum().sum())]
for g in ORDER:
    d_ = br[br.Group == g]; ids.append(g); labels.append(g); parents.append("root"); values.append(bn(d_["Passenger Kilometers"].sum())); cols.append(lf12[g])
    if d_.Airline.nunique() > 1:
        for a in d_.Airline: ids.append(f"{g}/{a}"); labels.append(a); parents.append(g); values.append(bn(d_.loc[d_.Airline == a, "Passenger Kilometers"].sum())); cols.append(lf12[g])
fig = go.Figure(go.Sunburst(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total", marker=dict(colors=cols, colorscale="Viridis", cmin=60, cmax=95, colorbar=dict(title=dict(text="Load factor %"))),
                            hovertemplate="%{label}<br>%{value:.1f} bn passenger-km<extra></extra>", insidetextorientation="radial"))
put("sunburst", 4, "Who flies the passenger-km, brand by brand?", "Inner ring = airline groups, outer ring = their brands. Angle = passenger-km over the last 12 months; colour = load factor.", fig,
    f"IndiGo flies {sh['IndiGo']:.0f}% of passenger-km and Air India Group {sh['Air India Group']:.0f}%; inside Air India Group, "
    + ", ".join(f"{r.Airline} {r._3 / br[br.Group == 'Air India Group']['Passenger Kilometers'].sum() * 100:.0f}%" for r in br[br.Group == "Air India Group"].sort_values("Passenger Kilometers", ascending=False).itertuples()) + ".", 520)

s = P["ask"]["IndiGo"]["2022-01-01":] / 1e6; q = s.groupby(s.index.to_period("Q"))
oh = pd.DataFrame({"o": q.first(), "h": q.max(), "l": q.min(), "c": q.last()}); xq = oh.index.to_timestamp()
fig = go.Figure(go.Candlestick(x=xq, open=oh.o, high=oh.h, low=oh.l, close=oh.c, increasing=dict(line=dict(color="#2166AC"), fillcolor="#2166AC"), decreasing=dict(line=dict(color="#B35806"), fillcolor="#FFFFFF"), name="IndiGo"))
fig.update_layout(xaxis_rangeslider_visible=False); fig.update_yaxes(title="IndiGo seat-km offered per month (billions)"); fig.update_xaxes(title="Quarter")
put("candle", 4, "How much capacity does IndiGo add each quarter?", "One candle per quarter: body = first to last month, wick = highest to lowest month. Filled blue = the quarter ended higher; hollow orange = lower.", fig,
    f"IndiGo's monthly seat-km went from {oh.o.iloc[0]:.1f} billion at the start of 2022 to {oh.c.iloc[-1]:.1f} billion; capacity ended higher than it started in {int((oh.c > oh.o).sum())} of {len(oh)} quarters.", 450)

# ---------------- SECTION 5: MOMENTUM AND MARKET SHARE ----------------
tm = P["rpk"].loc["2019-01-01":]; tt = tm.sum(axis=1); tmask = tt > 0
a, c2 = tm["IndiGo"] / tt * 100, tm["Air India Group"] / tt * 100
fig = go.Figure(go.Scatterternary(a=a[tmask], b=c2[tmask], c=(100 - a - c2)[tmask], mode="lines+markers", line=dict(color="#bbb", width=1),
        marker=dict(size=7, color=np.arange(tmask.sum()), colorscale="Viridis", showscale=True, colorbar=dict(title=dict(text="Months since<br>Jan 2019"))),
        text=[f"{d:%b %Y}" for d in tm.index[tmask]], hovertemplate="%{text}<br>IndiGo %{a:.0f}%<br>Air India Group %{b:.0f}%<br>Others %{c:.0f}%<extra></extra>"))
amin, bmin = max(0, np.floor(a[tmask].min() / 10) * 10 - 10), max(0, np.floor(c2[tmask].min() / 10) * 10 - 5)
cmin_ = max(0, min(np.floor((100 - a - c2)[tmask].min() / 10) * 10 - 5, 95 - amin - bmin))
fig.update_layout(ternary=dict(sum=100, aaxis=dict(title=dict(text="IndiGo %"), min=amin), baxis=dict(title=dict(text="Air India Group %"), min=bmin), caxis=dict(title=dict(text="All others %"), min=cmin_)))
put("ternary", 5, "How the three-way split of passenger-km moved", "Each dot is a month; a dot near a corner means that party dominates. Dark = early 2019, yellow = recent.", fig,
    f"IndiGo's share of passenger-km rose from {a['2019'].mean():.0f}% in 2019 to {a.iloc[-12:].mean():.0f}% in the last 12 months; Air India Group moved from {c2['2019'].mean():.0f}% to {c2.iloc[-12:].mean():.0f}% "
    f"and everyone else fell from {(100 - a - c2)['2019'].mean():.0f}% to {(100 - a - c2).iloc[-12:].mean():.0f}%.", 500)

ids, labels, parents, values, cols = ["root"], ["Scheduled domestic"], [""], [bn(br["Passenger Kilometers"].sum())], [float(PIL0["Index"].mean())]
for g in ORDER:
    d_ = br[br.Group == g]; ids.append(g); labels.append(g); parents.append("root"); values.append(bn(d_["Passenger Kilometers"].sum())); cols.append(float(PIL0.loc[g, "Index"]))
    if d_.Airline.nunique() > 1:
        for a_ in d_.Airline: ids.append(f"{g}/{a_}"); labels.append(a_); parents.append(g); values.append(bn(d_.loc[d_.Airline == a_, "Passenger Kilometers"].sum())); cols.append(float(PIL0.loc[g, "Index"]))
fig = go.Figure(go.Treemap(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total", marker=dict(colors=cols, colorscale="Viridis", cmin=0, cmax=100, colorbar=dict(title=dict(text="Index"))),
                           textinfo="label+percent root", hovertemplate="%{label}<br>%{value:.1f} bn passenger-km<extra></extra>"))
put("treemap", 5, "Is size the same as performance?", "Area = passenger-km over the last 12 months; colour = performance index (yellow = strong).", fig,
    f"IndiGo fills {sh['IndiGo']:.0f}% of the area at an index of {PIL0.loc['IndiGo', 'Index']:.0f}; Akasa Air covers {sh['Akasa Air']:.0f}% at {PIL0.loc['Akasa Air', 'Index']:.0f}. "
    f"The index rewards efficiency and momentum, not size.", 500)

by19 = pd.Series(b19["rpk"].sum(axis=1).values / 1e6, index=b19i.month).sort_index(); by12 = pd.Series(mk_rpk.loc[W12].values / 1e6, index=W12.month).sort_index()
fig = go.Figure([go.Barpolar(r=by12, theta=MON, name=f"{W12[0]:%b %Y} to {W12[-1]:%b %Y}", marker=dict(color="#2166AC"), opacity=.55),
                 go.Barpolar(r=by19, theta=MON, name="2019", marker=dict(color="#333333", line=dict(color="white", width=1)), opacity=.8, width=0.45)])
fig.update_layout(polar=dict(angularaxis=dict(direction="clockwise", rotation=90), radialaxis=dict(ticksuffix=" bn", angle=75, tickfont=dict(size=10)), barmode="overlay"))
gap = (by12 / by19 - 1) * 100
put("polarbar", 5, "Which months are busy?", "Each wedge is a calendar month; length = billion passenger-km. Narrow dark wedges = 2019, wide blue wedges = the last 12 months.", fig,
    f"The busiest month of the last 12 is {MON[by12.idxmax() - 1]} ({by12.max():.1f} billion passenger-km) and the quietest {MON[by12.idxmin() - 1]} ({by12.min():.1f} billion). "
    + (f"Every month beats 2019, by {gap.min():.0f}% to {gap.max():.0f}%." if gap.min() > 0 else f"{int((gap > 0).sum())} of 12 months beat 2019."), 500)

fc = forecast(mk_rpk, LAST); fx = pd.date_range(LAST + pd.DateOffset(months=1), periods=6, freq="MS"); band = fc.values * (MAPE.values / 100)
hist_ = mk_rpk.loc[LAST - pd.DateOffset(months=35):] / 1e6
px_ = [LAST] + list(fx); py = [mk_rpk.loc[LAST] / 1e6] + list(fc.values / 1e6)
up_ = [py[0]] + list((fc.values + band) / 1e6); lo_ = [py[0]] + list((fc.values - band) / 1e6)
fig = go.Figure([go.Scatter(x=px_ + px_[::-1], y=up_ + lo_[::-1], fill="toself", mode="lines", fillcolor="rgba(33,102,172,.18)", line=dict(width=0), name="Typical error band", hoverinfo="skip"),
                 go.Scatter(x=hist_.index, y=hist_, mode="lines", name="Actual", line=dict(color="#333", width=2.5)),
                 go.Scatter(x=px_, y=py, mode="lines+markers", name="Projection", line=dict(color="#2166AC", width=2.5, dash="dash"), marker=dict(size=8))])
fig.update_yaxes(title="Passenger-km per month (billions)", rangemode="tozero"); fig.update_xaxes(title="Month")
put("fan", 5, "What might the next six months look like?", "Projection = the same month last year x the latest 12-month growth rate. The band is the average error this rule made on past months.", fig,
    f"The rule projects {fc.iloc[0] / 1e6:.1f} billion passenger-km in {fx[0]:%B %Y} and {fc.iloc[-1] / 1e6:.1f} billion in {fx[-1]:%B %Y}. "
    f"Tested on {len(errs[1])} past months its average error was {MAPE[1]:.1f}% one month ahead and {MAPE[6]:.1f}% six months ahead: a rule of thumb, not a forecast model.", 450)


## 9. Sections 6 and 7: route network and international flights

In [9]:
# ---------------- SECTION 6: THE ROUTE NETWORK (city-pair data) ----------------
r19 = CR[CR.Date.dt.year == 2019].groupby(["p1", "p2"]).pax.sum(); rw = CR[CR.Date.isin(WC)].groupby(["p1", "p2"]).pax.sum()
RT = pd.concat([r19.rename("p19"), rw.rename("now")], axis=1).fillna(0); T19c, TNc = RT.p19.sum(), RT.now.sum()
RT["route"] = [f"{a.title()} - {b.title()}" for a, b in RT.index]

# 6.1 cross-source check
fig = go.Figure()
fig.add_hrect(y0=0.98, y1=1.02, fillcolor="#e8eef5", line_width=0, layer="below")
fig.add_trace(go.Scatter(x=XCHK.index, y=XCHK.values, mode="lines+markers", marker=dict(size=4, color="#2166AC"), line=dict(color="#2166AC", width=1.5), name="City-pair total / airline total",
                         hovertemplate="%{x|%b %Y}: %{y:.3f}<extra></extra>"))
fig.add_hline(y=1, line=dict(color="#222", dash="dash"))
fig.update_yaxes(title="City-pair / airline passengers", range=[0.9, 1.1]); fig.update_xaxes(title="Month")
x17, x18 = XCHK[XCHK.index.year == 2017], XCHK[XCHK.index.year >= 2018]
put("xchk", 6, "Do the two sources agree?", "Each point divides the month's city-pair passengers by the passengers in the airline files. 1.00 = identical; the shaded band is +/-2%.", fig,
    f"From 2018 the two sources agree almost exactly (median ratio {x18.median():.3f}, {(x18.sub(1).abs() <= 0.02).mean() * 100:.0f}% of months within 2%). "
    f"In 2017 the city-pair total is {(x17.min() - 1) * 100:.0f}% to {(x17.max() - 1) * 100:.0f}% higher: the airline files lack AirAsia India's 2017 workbook, which the download could not fetch (HTTP 403).", 380)

# 6.2 route-type recovery
mkc = CR.groupby(["Date", "kind"]).pax.sum().unstack(); bb = mkc[mkc.index.year == 2019]; bb.index = bb.index.month
rat = (mkc / bb.reindex(mkc.index.month).values * 100).rolling(3).mean().loc["2019-03-01":]
KC = {"Metro to metro": "#B35806", "Metro to other city": "#2166AC", "Other city to other city": "#009E73"}; KD = {"Metro to metro": "solid", "Metro to other city": "dash", "Other city to other city": "dot"}
fig = go.Figure([go.Scatter(x=rat.index, y=rat[k], name=k, mode="lines", line=dict(color=KC[k], width=2.5, dash=KD[k])) for k in KC])
fig.add_hline(y=100, line=dict(color="#222", width=1)); fig.update_yaxes(title="% of the same months in 2019"); fig.update_xaxes(title="Month")
kk = RT.join(CR.drop_duplicates(["p1", "p2"]).set_index(["p1", "p2"])["kind"]).groupby("kind")[["p19", "now"]].sum()
first_back = {k: rat.loc["2020-07-01":, k][lambda s: s >= 100].index.min() for k in KC}
put("routetype", 6, "Which kinds of routes came back first?", f"Routes grouped by their two ends; metros = {', '.join(m.title() for m in METRO)}. Lines are 3-month averages against the same months of 2019.", fig,
    f"Over the last 12 months metro-to-metro routes carry {kk.loc['Metro to metro', 'now'] / kk.loc['Metro to metro', 'p19'] * 100:.0f}% of their 2019 passengers, metro-to-other {kk.loc['Metro to other city', 'now'] / kk.loc['Metro to other city', 'p19'] * 100:.0f}% "
    f"and other-to-other {kk.loc['Other city to other city', 'now'] / kk.loc['Other city to other city', 'p19'] * 100:.0f}%. Routes between smaller cities got back to 2019 first ({first_back['Other city to other city']:%b %Y}); metro-to-metro last ({first_back['Metro to metro']:%b %Y}).", 460)

# 6.3 metro cities, year by year (heat table)
CT = pd.concat([CR.rename(columns={"p1": "city"})[["Date", "city", "pax"]], CR.rename(columns={"p2": "city"})[["Date", "city", "pax"]]])
CT["city2"] = np.where(CT.city.isin(METRO), CT.city.str.title(), "All other cities")
cy = CT.groupby(["city2", "Date"]).pax.sum().unstack(0)
yrs = list(range(2019, LASTC.year + 1)); rows = {}
for y in yrs:
    mm = cy.index[(cy.index.year == y)]; base = cy[cy.index.year == 2019]; base = base[base.index.month.isin(mm.month)]
    rows[y] = cy.loc[mm].sum() / base.sum() * 100
HM = pd.DataFrame(rows); HM = HM.loc[[m.title() for m in METRO] + ["All other cities"]]
lbl = [str(y) if y < LASTC.year else f"{y} (Jan-{LASTC:%b})" for y in yrs]
fig = go.Figure(go.Heatmap(z=HM.values, x=lbl, y=HM.index, colorscale=DIV, zmid=100, zmin=40, zmax=160, text=HM.round(0).values, texttemplate="%{text:.0f}", xgap=2, ygap=2,
                           colorbar=dict(title=dict(text="% of 2019")), hovertemplate="%{y}, %{x}: %{z:.0f}% of the same months in 2019<extra></extra>"))
fig.update_yaxes(autorange="reversed")
last_col = HM.iloc[:, -1]
put("metroheat", 6, "How did traffic at the six metros grow, year by year?", "Each cell = passengers on all routes touching that city, as % of the same months of 2019. Blue = above 2019, orange = below, grey = equal.", fig,
    f"In {lbl[-1]} {last_col.idxmax()} is at {last_col.max():.0f}% of 2019 and {last_col.idxmin()} at {last_col.min():.0f}%, while the other cities together reach {last_col['All other cities']:.0f}%. "
    f"2020 shows how deep the fall was ({HM[2020].min():.0f}% to {HM[2020].max():.0f}%).", 400)

# 6.4 dominant routes, 2019 vs now (dumbbell)
topR = RT.sort_values("p19", ascending=False).head(12)
fig = go.Figure()
for r in topR.itertuples():
    fig.add_trace(go.Scatter(x=[r.p19 / 1e6, r.now / 1e6], y=[r.route, r.route], mode="lines", line=dict(color="#c8c8c8", width=5), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=topR.p19 / 1e6, y=topR.route, mode="markers", name="2019", marker=dict(symbol="circle-open", size=12, color="#333", line=dict(width=2.5))))
fig.add_trace(go.Scatter(x=topR.now / 1e6, y=topR.route, mode="markers", name=f"{WC[0]:%b %Y} to {WC[-1]:%b %Y}", marker=dict(symbol="diamond", size=12, color=["#2166AC" if v >= 1 else "#B35806" for v in topR.now / topR.p19])))
fig.add_trace(go.Scatter(x=topR[["p19", "now"]].max(axis=1) / 1e6 + 0.12, y=topR.route, mode="text", text=[f"{v:.0f}%" for v in topR.now / topR.p19 * 100], textposition="middle right", showlegend=False, hoverinfo="skip"))
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title="Passengers in 12 months (millions, both directions)"); fig.update_layout(margin=dict(l=170))
up_ = int((topR.now >= topR.p19).sum())
put("toproutes", 6, "Did the dominant routes grow?", "The 12 busiest routes of 2019. Circle = 2019, diamond = last 12 months (blue = above 2019, orange = below); the label is % of 2019.", fig,
    f"{up_} of the 12 biggest 2019 routes are above their 2019 level. {topR.route.iloc[0]}, the busiest, carries {topR.now.iloc[0] / topR.p19.iloc[0] * 100:.0f}% of its 2019 passengers. "
    f"Together these routes went from {topR.p19.sum() / T19c * 100:.1f}% of all passengers to {topR.now.sum() / TNc * 100:.1f}%.", 520)

# 6.5 route churn waterfall
lost = RT[(RT.p19 > 0) & (RT.now == 0)]; new = RT[(RT.p19 == 0) & (RT.now > 0)]; kept = RT[(RT.p19 > 0) & (RT.now > 0)]
steps = [T19c / 1e6, -lost.p19.sum() / 1e6, (kept.now.sum() - kept.p19.sum()) / 1e6, new.now.sum() / 1e6]
fig = go.Figure(go.Waterfall(x=["2019", f"Routes dropped ({len(lost)})", f"Growth on routes kept ({len(kept)})", f"New routes ({len(new)})", "Last 12 months"],
        measure=["absolute", "relative", "relative", "relative", "total"], y=steps + [0], text=[f"{steps[0]:.1f}", f"{steps[1]:+.1f}", f"{steps[2]:+.1f}", f"{steps[3]:+.1f}", f"{TNc / 1e6:.1f}"],
        textposition="outside", increasing=dict(marker=dict(color="#2166AC")), decreasing=dict(marker=dict(color="#B35806")), totals=dict(marker=dict(color="#555"))))
fig.update_yaxes(title="Passengers in 12 months (millions)", range=[0, TNc / 1e6 * 1.2])
nb_ = new.sort_values("now", ascending=False).head(3)
put("churn", 6, "Where did the extra passengers come from?", "From 2019 passengers to the last 12 months in three steps: routes no longer flown, growth on routes flown in both periods, and routes that did not exist in 2019.", fig,
    f"Dropped routes cost {lost.p19.sum() / 1e6:.1f} million passengers ({lost.p19.sum() / T19c * 100:.1f}% of 2019), routes flown in both periods added {steps[2]:+.1f} million and new routes {steps[3]:+.1f} million. "
    f"The biggest new routes are {', '.join(nb_.route.str.title())}.", 440)

# 6.6 distribution of route recovery
kb = kept[kept.p19 >= 100_000]; rr = (kb.now / kb.p19 * 100).clip(upper=300)
fig = go.Figure(go.Histogram(x=rr, xbins=dict(start=0, end=310, size=10), marker=dict(color="#2166AC", line=dict(color="white", width=1)), hovertemplate="%{x}%: %{y} routes<extra></extra>"))
fig.add_vline(x=100, line=dict(color="#222", dash="dash"), annotation_text="2019 level", annotation_position="top")
fig.add_vline(x=float(rr.median()), line=dict(color="#B35806"), annotation_text=f"median {rr.median():.0f}%", annotation_position="top right")
fig.update_xaxes(title="Passengers in the last 12 months, % of 2019 (capped at 300%)"); fig.update_yaxes(title="Number of routes")
put("routehist", 6, "How many routes are back above 2019?", "Routes that had at least 100,000 passengers in 2019 and are still flown. Each bar counts routes in a 10-point band.", fig,
    f"{(rr >= 100).mean() * 100:.0f}% of these {len(rr)} routes are above their 2019 level; the median route is at {rr.median():.0f}% and the middle half runs from {rr.quantile(.25):.0f}% to {rr.quantile(.75):.0f}%.", 420)

# 6.7 Lorenz curve: how concentrated is traffic on a few routes?
def lorenz(v):
    v = np.sort(v[v > 0]); cx = np.r_[0, np.arange(1, len(v) + 1) / len(v)]; cy_ = np.r_[0, np.cumsum(v) / v.sum()]
    return cx, cy_, 1 - 2 * np.trapezoid(cy_, cx) if hasattr(np, "trapezoid") else 1 - 2 * np.trapz(cy_, cx)
x1, y1, g19 = lorenz(RT.p19.values); x2, y2, gnw = lorenz(RT.now.values)
top10 = lambda v: np.sort(v[v > 0])[::-1][:max(1, int(len(v[v > 0]) * .1))].sum() / v.sum() * 100
fig = go.Figure([go.Scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(color="#aaa", dash="dot"), name="Every route equal"),
                 go.Scatter(x=x1 * 100, y=y1 * 100, mode="lines", line=dict(color="#333", width=2.5), name=f"2019 (Gini {g19:.2f})"),
                 go.Scatter(x=x2 * 100, y=y2 * 100, mode="lines", line=dict(color="#2166AC", width=2.5, dash="dash"), name=f"Last 12 months (Gini {gnw:.2f})")])
fig.update_xaxes(title="Routes, from smallest to largest (cumulative %)"); fig.update_yaxes(title="Passengers (cumulative %)")
put("lorenz", 6, "Is traffic concentrated on a few routes?", "A Lorenz curve: the further the curve sags below the diagonal, the more traffic sits on a few big routes. Gini 0 = all routes equal, 1 = one route has everything.", fig,
    f"The busiest 10% of routes carried {top10(RT.p19.values):.0f}% of passengers in 2019 and {top10(RT.now.values):.0f}% in the last 12 months (Gini {g19:.2f} to {gnw:.2f}). "
    f"Route concentration barely moved even as the number of routes grew, while concentration among airlines rose sharply.", 460)

# 6.8 cities: diverging bar against 2019
cty = CT.assign(y19=CT.Date.dt.year == 2019, w=CT.Date.isin(WC)).groupby("city").apply(lambda d: pd.Series({"p19": d.loc[d.y19, "pax"].sum(), "now": d.loc[d.w, "pax"].sum()}))
cty = cty[cty.p19 >= 500_000]; cty["pct"] = cty.now / cty.p19 * 100 - 100; cty = cty.sort_values("pct")
fig = go.Figure(go.Bar(x=cty.pct, y=[f"<b>{c.title()}</b>" if c in METRO else c.title() for c in cty.index], orientation="h",
        marker=dict(color=["#2166AC" if v >= 0 else "#B35806" for v in cty.pct], pattern=dict(shape=["" if v >= 0 else "/" for v in cty.pct])),
        hovertemplate="%{y}: %{x:+.0f}% vs 2019<extra></extra>"))
fig.add_vline(x=0, line=dict(color="#222", width=1)); fig.update_xaxes(title="Passengers in the last 12 months vs 2019 (%)"); fig.update_yaxes(tickfont=dict(size=10))
fig.update_layout(margin=dict(l=150))
put("cities", 6, "Which cities are above or below 2019?", "Every city with at least 0.5 million passengers in 2019 (all routes touching it). Metros in bold; hatched orange = below 2019.", fig,
    f"{int((cty.pct < 0).sum())} of {len(cty)} cities are still below 2019, led by {cty.index[0].title()} ({cty.pct.iloc[0]:+.0f}%); {cty.index[-1].title()} grew most ({cty.pct.iloc[-1]:+.0f}%). "
    f"Among the metros, {cty.loc[cty.index.isin(METRO)].pct.idxmax().title()} grew most ({cty.loc[cty.index.isin(METRO)].pct.max():+.0f}%) and {cty.loc[cty.index.isin(METRO)].pct.idxmin().title()} least ({cty.loc[cty.index.isin(METRO)].pct.min():+.0f}%).", 900)

# ---------------- SECTION 7: INTERNATIONAL FLIGHTS (Indian airlines) ----------------
bi19 = PI["rpk"].loc[b19i]
IBASE = [g for g in PI["rpk"].columns if (bi19[g] > 0).sum() == 12 and bi19[g].sum() >= 0.005 * bi19.sum().sum() and PI["rpk"].loc[W12, g].fillna(0).sum() > 0]   # still flying
dom = {g: P["rpk"].loc[W12, g].sum() / b19["rpk"][g].sum() * 100 for g in IBASE if g in P["rpk"].columns and b19["rpk"][g].sum() > 0}
intl = {g: PI["rpk"].loc[W12, g].fillna(0).sum() / bi19[g].sum() * 100 for g in IBASE}
gI = sorted(dom, key=lambda g: -intl[g])
fig = go.Figure()
for g in gI:
    fig.add_trace(go.Scatter(x=[dom[g], intl[g]], y=[g, g], mode="lines", line=dict(color="#c8c8c8", width=5), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=[dom[g] for g in gI], y=gI, mode="markers+text", name="Domestic", marker=dict(symbol="circle", size=13, color="#333"), text=[f"{dom[g]:.0f}%" for g in gI], textposition="bottom center"))
fig.add_trace(go.Scatter(x=[intl[g] for g in gI], y=gI, mode="markers+text", name="International", marker=dict(symbol="diamond", size=13, color="#2166AC"), text=[f"{intl[g]:.0f}%" for g in gI], textposition="top center"))
fig.add_vline(x=100, line=dict(color="#222", dash="dash")); fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title="Passenger-km in the last 12 months, % of 2019"); fig.update_layout(margin=dict(l=130))
put("intl_dumbbell", 7, "Domestic vs international recovery", "Airlines that flew scheduled international routes in every month of 2019 and still fly them (Go First stopped in 2023). Circle = domestic, diamond = international, both as % of their own 2019 passenger-km.", fig,
    "; ".join(f"{g}: domestic {dom[g]:.0f}%, international {intl[g]:.0f}%" + (" (stopped flying)" if P["pax"].loc[W12, g].fillna(0).sum() == 0 else "") for g in gI) + f". International passenger-km for all Indian airlines is at {PI['rpk'].loc[W12].sum().sum() / bi19.sum().sum() * 100:.0f}% of 2019.", 380)

ir = pd.DataFrame({g: PI["rpk"][g] / pd.Series(months.month, index=months).map(bi19.set_axis(b19i.month)[g]) * 100 for g in IBASE}).rolling(3).mean().loc["2019-03-01":]
fig = go.Figure([go.Scatter(x=ir.index, y=ir[g], name=g, mode="lines", line=dict(color=gc(g), width=2.5), connectgaps=False) for g in IBASE])
fig.add_vrect(x0="2020-04-01", x1="2022-03-01", fillcolor="#eeeeee", line_width=0, layer="below", annotation_text="scheduled international flights suspended", annotation_position="top left")
fig.add_hline(y=100, line=dict(color="#222", dash="dash")); fig.update_yaxes(title="% of the same months in 2019"); fig.update_xaxes(title="Month")
fbI = {g: ir.loc["2022-04-01":, g][lambda s: s >= 100].index.min() for g in IBASE}
put("intl_lines", 7, "International passenger-km against 2019", "3-month averages for each airline's scheduled international flights. The grey band is the period with no scheduled international flights.", fig,
    "; ".join(f"{g} first back to 2019 in {d:%b %Y}" if pd.notna(d) else f"{g} not back to 2019" for g, d in fbI.items()) + ".", 440)


## 10. Section 8: relationships and prediction
The correlation matrix is interactive in the HTML page (click a cell). `pair_fig(x, y)` draws any pair here, for example `pair_fig('gro', 'shm').show()`.

In [10]:
# ---------------- SECTION 8: RELATIONSHIPS AND PREDICTION ----------------
KEYS = list(MET); LBL = {k: MET[k][0] for k in KEYS}
CM = pd.DataFrame(np.nan, index=KEYS, columns=KEYS); CN = CM.copy()
for a_ in KEYS:
    for b_ in KEYS:
        if a_ == b_: CN.loc[a_, b_] = OBS[a_].notna().sum(); CM.loc[a_, b_] = 1.0; continue
        d_ = OBS[[a_, b_]].dropna(); CN.loc[a_, b_] = len(d_)
        if len(d_) >= 20: CM.loc[a_, b_] = spearman(d_[a_], d_[b_])
pairs = [(a_, b_, CM.loc[a_, b_]) for i, a_ in enumerate(KEYS) for b_ in KEYS[i + 1:] if MET[a_][3] != MET[b_][3] and pd.notna(CM.loc[a_, b_]) and CN.loc[a_, b_] >= 30]
full_n = int(CN.values.max() * 0.9)
INTERESTING = (sorted([t for t in pairs if CN.loc[t[0], t[1]] >= full_n], key=lambda t: -abs(t[2]))[:2]      # strongest cross-pillar pairs covering all airlines
               + sorted([t for t in pairs if CN.loc[t[0], t[1]] < full_n], key=lambda t: -abs(t[2]))[:2])    # strongest pairs that involve the 2019 yardstick

def pair_fig(xk, yk):
    f = go.Figure()
    for g in ORDER + [g for g in OBS.g.unique() if g not in ORDER]:
        d_ = OBS[(OBS.g == g)][[xk, yk, "T"]].dropna()
        if len(d_): f.add_trace(go.Scatter(x=d_[xk], y=d_[yk], mode="markers", name=g, marker=dict(color=gc(g), symbol=gs(g), size=8, line=dict(color="white", width=.5)),
                                           text=d_["T"].dt.strftime("%b %Y"), hovertemplate=g + "<br>%{text}<br>x %{x:.1f}, y %{y:.1f}<extra></extra>"))
    f.update_xaxes(title=f"{MET[xk][0]} ({SHORTU[xk]})"); f.update_yaxes(title=f"{MET[yk][0]} ({SHORTU[yk]})")
    f.update_layout(template="plotly_white", height=430, margin=dict(l=70, r=20, t=50, b=60), legend=dict(orientation="h", y=-0.25),
                    title=dict(text=f"{MET[xk][0]} vs {MET[yk][0]}: rank correlation {CM.loc[xk, yk]:+.2f} (n = {int(CN.loc[xk, yk])})", font=dict(size=13), x=0))
    return f

zc = CM.values.astype(float); tc = [[("" if np.isnan(v) else f"{v:+.2f}") for v in row] for row in zc]
fig = go.Figure(go.Heatmap(z=zc, x=[LBL[k] for k in KEYS], y=[LBL[k] for k in KEYS], zmin=-1, zmax=1, colorscale=[[0, "#B35806"], [0.5, "#F2F2F2"], [1, "#2166AC"]],
                           text=tc, texttemplate="%{text}", xgap=2, ygap=2, customdata=CN.values, colorbar=dict(title=dict(text="Rank<br>correlation")),
                           hovertemplate="%{y} vs %{x}<br>rank correlation %{z:.2f} (n = %{customdata})<br><b>click to plot this pair</b><extra></extra>"))
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(tickangle=-35); fig.update_layout(margin=dict(l=190, b=120))
desc = lambda t: f"{LBL[t[0]].lower()} and {LBL[t[1]].lower()} ({t[2]:+.2f})"
put("corr", 8, "Which metrics move together? Click any cell to plot that pair", "Rank correlation between two metrics across every airline-month since mid-2023 (blue = rise together, orange = one rises as the other falls). Click a cell, or a button below the chart, to draw that pair.", fig,
    "Interesting pairs (strongest links between metrics from different pillars; the first two cover all airlines, the last two only those with a 2019 yardstick): " + "; ".join(desc(t) for t in INTERESTING) +
    ". Pairs inside one pillar, such as demand and capacity recovery, are high partly because they are built from the same columns.", 560)
FIGS["corr"]["pairs"] = INTERESTING

# distance between airlines: what 'similar' means
XS = S0.loc[ORDER, COMMON].fillna(50)
ZL = linkage(XS.values, "ward"); leaf = [ORDER[i] for i in dendrogram(ZL, no_plot=True, labels=None)["leaves"]]
DM = pd.DataFrame(squareform(pdist(XS.values)), index=ORDER, columns=ORDER).loc[leaf, leaf]
fig = go.Figure(go.Heatmap(z=DM.values, x=leaf, y=leaf, colorscale=[[0, "#08306b"], [0.5, "#6baed6"], [1, "#f7fbff"]], text=DM.round(0).values, texttemplate="%{text:.0f}",
                           xgap=2, ygap=2, colorbar=dict(title=dict(text="Distance")), hovertemplate="%{y} to %{x}: distance %{z:.0f}<extra></extra>"))
fig.update_yaxes(autorange="reversed"); fig.update_layout(margin=dict(l=130))
off = DM.where(~np.eye(len(DM), dtype=bool)).stack(); cl_, far_ = off.idxmin(), off.idxmax()
put("distance", 8, "How far apart are the airlines?", "Distance = square root of the sum of squared differences in the six metric scores every airline has (load factor, passengers per departure, productivity, growth, share momentum, volatility). Two airlines 25 points apart on all six are sqrt(6 x 25^2) = 61 apart. Dark = similar.", fig,
    f"The closest pair is {cl_[0]} and {cl_[1]} (distance {off[cl_]:.0f}); the furthest apart are {far_[0]} and {far_[1]} ({off[far_]:.0f}). "
    f"Scores are used rather than raw values so that each metric counts equally, whatever its unit.", 470)

THR = (ZL[-3, 2] + ZL[-2, 2]) / 2
fig = ff.create_dendrogram(XS.values, labels=ORDER, orientation="left", linkagefun=lambda x: linkage(x, "ward"), color_threshold=THR,
                           colorscale=["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#A0522D", "#333333"])
fig.add_vline(x=THR, line=dict(dash="dot", color="#888")); fig.update_xaxes(title="Ward linkage height (how much grouping these airlines increases the spread within the group)")
groups_ = [[g for g, k in zip(ORDER, fcluster(ZL, t=3, criterion="maxclust")) if k == c] for c in sorted(set(fcluster(ZL, t=3, criterion="maxclust")))]
put("dendro", 8, "Which airlines form peer groups?", "Built from the same distances. Airlines (or groups) are joined step by step, always merging the pair that adds the least spread; the dotted line cuts the tree into three peer groups.", fig,
    "Three peer groups: " + "; ".join(", ".join(v) for v in groups_) + ". The split follows business model (big jets on long trips vs small aircraft on short hops) more than recovery.", 430)

lab = {**LBL, **{p: p + " pillar" for p in PILLARS}, "Index": "Performance index"}
ps = PRED.sort_values(); xs, ys = [], []
for k, v in ps.items(): xs += [0, v, None]; ys += [lab[k], lab[k], None]
fig = go.Figure([go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#bbb", width=3), hoverinfo="skip", showlegend=False),
                 go.Scatter(x=ps.values, y=[lab[k] for k in ps.index], mode="markers+text", text=[f"{v:+.2f}" for v in ps.values], textposition=["middle right" if v >= 0 else "middle left" for v in ps.values],
                            marker=dict(size=13, color=["#2166AC" if v > .3 else ("#B35806" if v < -.1 else "#999") for v in ps.values],
                                        symbol=["circle" if v > .3 else ("x" if v < -.1 else "circle-open") for v in ps.values], line=dict(color="#222", width=1)),
                            customdata=[int(NOBS[k]) for k in ps.index], hovertemplate="%{y}: %{x:+.2f} (n = %{customdata})<extra></extra>", showlegend=False)])
fig.add_vline(x=0, line=dict(color="#222", width=1)); fig.update_xaxes(title="Rank correlation with passenger-km growth over the next 6 months", range=[-.7, 1.05])
bestm = ps.drop(["Index"] + PILLARS).idxmax(); worstm = ps.drop(["Index"] + PILLARS).idxmin()
put("lollipop", 8, "Which metrics predict what happens next?", "How strongly each measure, taken at a month, lines up with passenger-km growth over the following 6 months (+1 = perfectly). Signed so that + means the metric's 'better' direction goes with faster growth.", fig,
    f"{LBL[bestm]} is the best guide ({ps[bestm]:+.2f}) and {LBL[worstm].lower()} the worst ({ps[worstm]:+.2f}); the index scores {ps['Index']:+.2f}. "
    f"Based on {len(OBS)} airline-months with overlapping windows, so treat these as indicative.", 560)

labs = PILLARS + ["Index"]; peers = BT_P.drop(index="Go First"); xs, ys = [], []
for p in labs:
    v = peers[p].dropna(); xs += [v.min(), v.max(), None]; ys += [p, p, None]
fig = go.Figure([go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#ccc", width=12), name="Range of the other airlines", hoverinfo="skip"),
                 go.Scatter(x=[peers[p].median() for p in labs], y=labs, mode="markers", marker=dict(symbol="line-ns", size=18, color="#555", line=dict(width=3, color="#555")), name="Peer median"),
                 go.Scatter(x=[BT_P.loc["Go First", p] for p in labs], y=labs, mode="markers+text", text=[f"{BT_P.loc['Go First', p]:.0f}" for p in labs], textposition="top center",
                            marker=dict(size=15, color=gc("Go First"), symbol="diamond", line=dict(color="#222", width=1)), name="Go First")])
fig.update_xaxes(title=f"Score as of {BT_T:%B %Y}, using only data up to then", range=[0, 100]); fig.update_yaxes(autorange="reversed")
sp = BT_P["Index"].drop("Go First").sort_values()
put("backtest", 8, "Would the metrics have flagged Go First?", "Grey bar = range of the other airlines' scores in the month Go First stopped flying; dark tick = their median; diamond = Go First.", fig,
    f"As of {BT_T:%B %Y} Go First ranked {BT_rank['Go First']} of {len(BT_P)} on the index ({BT_P.loc['Go First', 'Index']:.0f}), with momentum at {BT_P.loc['Go First', 'Momentum']:.0f}. "
    f"The next-lowest airline, {sp.index[0]}, scored {sp.iloc[0]:.0f} and is still flying: traffic data shows weak momentum but cannot see a funding crisis.", 430)


## 11. Conclusions

In [11]:
# CONCLUSIONS: every number below is computed from the data above
tot19, tot12m = mk.loc[b19i].mean() / 1e6, mk.loc[W12].mean() / 1e6
hhi = lambda sh_: float((sh_ ** 2).sum() * 1e4)
sr19 = b19["rpk"].sum() / b19["rpk"].sum().sum(); sr12 = P["rpk"].loc[W12].sum() / P["rpk"].loc[W12].sum().sum()
REC = pd.DataFrame({g: dict(when=ROLL.loc["2020-07-01":, g].dropna()[lambda s: s >= 1].index.min()) for g in BASE}).T
firstg = REC.dropna(subset=["when"]).sort_values("when"); never = REC[REC.when.isna()]
lf_rows = [(g, M0.loc[g, "lfd"]) for g in BASE if g in M0.index and pd.notna(M0.loc[g, "lfd"])]
lf_back = [g for g, v in lf_rows if v >= 0]
bestp, worstp = PRED.drop(["Index"] + PILLARS).idxmax(), PRED.drop(["Index"] + PILLARS).idxmin()
second = PRED.drop(["Index"] + PILLARS + [bestp]).idxmax()
CONCL = [
  ("Passenger numbers: the market is back, but only two of six 2019 airlines are above 2019.",
   f"Passengers per month averaged {tot12m:.1f} million over the last 12 months against {tot19:.1f} million in 2019 ({(tot12m / tot19 - 1) * 100:+.0f}%; passenger-km {(RPK12 / RPK19 - 1) * 100:+.0f}%). "
   f"Of the {len(BASE)} airlines that flew all of 2019, {len(firstg)} regained their 2019 passenger level (" + "; ".join(f"{g} in {d:%B %Y}" for g, d in firstg["when"].items())
   + f"), {sum(1 for g in BASE if STATUS[g] == 'Recovered and holding')} are still above it, and {', '.join(never.index)} never got back."),
  ("Load factors: almost nobody is back to 2019.",
   (f"Only {' and '.join(lf_back)} {'fills' if len(lf_back) == 1 else 'fill'} as large a share of seats as in 2019" if lf_back else "No airline fills as large a share of seats as in 2019")
   + " (" + "; ".join(f"{g} {v:+.1f} points" for g, v in sorted(lf_rows, key=lambda t: -t[1])) + "). Capacity came back faster than demand, so planes are a little emptier even where passengers grew."),
  ("The market concentrated on airlines but not on routes.",
   f"IndiGo's share of passenger-km went from {sr19['IndiGo'] * 100:.0f}% to {sr12['IndiGo'] * 100:.0f}% and the airline concentration index (HHI) from {hhi(sr19):,.0f} to {hhi(sr12):,.0f} (above 2,500 is highly concentrated). "
   f"On routes, the busiest 10% carry {top10(RT.p19.values):.0f}% of passengers then and {top10(RT.now.values):.0f}% now: growth came from many routes, including {len(new)} new ones."),
  ("Recovery was led by smaller cities, not the metros.",
   f"Metro-to-metro routes are at {kk.loc['Metro to metro', 'now'] / kk.loc['Metro to metro', 'p19'] * 100:.0f}% of 2019 while routes between smaller cities are at {kk.loc['Other city to other city', 'now'] / kk.loc['Other city to other city', 'p19'] * 100:.0f}%. "
   f"{int((cty.pct < 0).sum())} of {len(cty)} large cities are still below 2019, including {cty.index[0].title()} ({cty.pct.iloc[0]:+.0f}%) and Kolkata ({cty.loc['KOLKATA', 'pct']:+.0f}%)."),
  ("International flying is well above 2019, driven by IndiGo.",
   f"Indian airlines' international passenger-km is at {PI['rpk'].loc[W12].sum().sum() / bi19.sum().sum() * 100:.0f}% of 2019. IndiGo's is at {intl['IndiGo']:.0f}% against {dom['IndiGo']:.0f}% at home; "
   f"Air India Group's at {intl['Air India Group']:.0f}% against {dom['Air India Group']:.0f}% at home, and SpiceJet's at {intl['SpiceJet']:.0f}%."),
  ("The index, with countermeasures for new airlines, puts IndiGo first.",
   f"IndiGo scores {PIL0.loc['IndiGo', 'Index']:.0f}, Akasa Air {PIL0.loc['Akasa Air', 'Index']:.0f}, Air India Group {PIL0.loc['Air India Group', 'Index']:.0f} and {bot} {PIL0.loc[bot, 'Index']:.0f}. "
   f"Without the countermeasures Akasa Air would lead ({PILN.loc['Akasa Air', 'Index']:.0f}) because it was scored only on its two strongest pillars. No airline moves more than {int(swing.max())} places when a pillar's weight is doubled."),
  ("Growth predicts growth; efficiency does not.",
   f"{LBL[bestp]} lines up best with the next six months ({PRED[bestp]:+.2f}), then {LBL[second].lower()} ({PRED[second]:+.2f}); the Operations pillar scores {PRED['Operations']:+.2f}. "
   f"In a backtest Go First ranked last of {len(BT_P)} in its final month (index {BT_P.loc['Go First', 'Index']:.0f}), but the margin over SpiceJet was small: a warning light, not a verdict."),
]
LIMITS = ["Scores are relative: 50 means typical of the airlines scored that month, not an absolute standard. Only seven airlines are scored.",
          "Operations metrics favour big jets on long trips; turboprop operators (Star Air, Fly91, Alliance Air) score low there regardless of management.",
          "The countermeasure settings (floors, 36-month maturity, 50 for a missing pillar) are judgment calls; the old-vs-adjusted chart shows their effect.",
          "City-pair data has no airline column, and some city merges are judgment calls (two Goa airports, Navi Mumbai into Mumbai, Hindon into Ghaziabad).",
          "Months with no scheduled international flights (April 2020 to early 2022) are treated as zero; air-bubble flights in that period are not in this series.",
          f"DGCA data is provisional. Airline analysis stops at {LAST:%B %Y} (last month reconciling to DGCA's total within 2%); city-pair data ends {LASTC:%B %Y}."]
for t, b_ in CONCL: print("-", t, b_, "\n")


- Passenger numbers: the market is back, but only two of six 2019 airlines are above 2019. Passengers per month averaged 13.9 million over the last 12 months against 12.0 million in 2019 (+16%; passenger-km +18%). Of the 6 airlines that flew all of 2019, 3 regained their 2019 passenger level (IndiGo in May 2022; Air India Group in December 2022; Alliance Air in December 2022), 2 are still above it, and Go First, SpiceJet, TruJet never got back. 

- Load factors: almost nobody is back to 2019. Only Air India Group fills as large a share of seats as in 2019 (Air India Group +0.6 points; IndiGo -2.9 points; Alliance Air -3.3 points; SpiceJet -7.3 points). Capacity came back faster than demand, so planes are a little emptier even where passengers grew. 

- The market concentrated on airlines but not on routes. IndiGo's share of passenger-km went from 47% to 62% and the airline concentration index (HHI) from 3,153 to 4,682 (above 2,500 is highly concentrated). On routes, the busiest 10% car

## 12. Build the HTML page

In [12]:
# ASSEMBLE ONE SELF-CONTAINED HTML PAGE (Plotly.js embedded, so it opens offline)
esc = _html.escape
CFG = dict(displaylogo=False, responsive=True, modeBarButtonsToRemove=["lasso2d", "select2d"])
kpis = [(f"{(RPK12 / RPK19 - 1) * 100:+.0f}%", "passenger-km vs 2019", f"passengers {(tot12m / tot19 - 1) * 100:+.0f}%, last 12 months"),
        (f"{top}", "highest performance index", f"{PIL0.loc[top, 'Index']:.0f} of 100, all three pillars"),
        (f"{kk.loc['Metro to metro', 'now'] / kk.loc['Metro to metro', 'p19'] * 100:.0f}% vs {kk.loc['Other city to other city', 'now'] / kk.loc['Other city to other city', 'p19'] * 100:.0f}%", "metro-metro vs small-city routes", "passengers as % of 2019"),
        (f"{LBL[bestp]}", "best predictor of the next 6 months", f"rank correlation {PRED[bestp]:+.2f}")]
kpi_html = "".join(f'<div class="kpi"><div class="v">{esc(a)}</div><div class="l">{esc(b_)}</div><div class="s">{esc(c)}</div></div>' for a, b_, c in kpis)
met_rows = "".join(f"<tr><td>{p}</td><td><b>{esc(MET[k][0])}</b><div class='small'>{esc(WHY[k])}</div></td><td>{esc(FORMULA[k][0])}</td><td>{esc(FORMULA[k][1])}</td>"
                   f"<td>{esc(MET[k][1])}</td><td>{'higher' if MET[k][2] else 'lower'}</td><td class='mono'>{esc(WORKED.get(k, 'n/a'))}</td></tr>" for p in PILLARS for k in MET if MET[k][3] == p)
fl = "".join(f"<li><b>{esc(t)}</b> {esc(d)}</li>" for t, d in [
    ("Two full years before momentum counts.", "Growth, share momentum and volatility compare the last 12 months with the 12 before, so an airline needs 24 months of traffic before they exist."),
    ("Log growth instead of % growth.", f"A ramp-up from a small base produces huge % growth. ln(this year / last year) is symmetric and compresses it: Fly91 is {MN.loc['Fly91', 'gro']:+.0f}% but {M0.loc['Fly91', 'gro']:+.0f} log points."),
    ("Launch months left out of volatility.", "A month is used only if the same month a year earlier had at least 25,000 passengers, so the launch ramp does not count as erratic traffic."),
    ("Momentum damped for young airlines.", f"Momentum scores are pulled toward 50 in proportion to age: full weight after {MATURE} months. Fly91 ({age_at('Fly91', LAST)} months) gets {min(1, age_at('Fly91', LAST) / MATURE):.0%} weight."),
    ("Robust scaling.", "Each metric is centred on the median and scaled by the median absolute deviation, with a minimum meaningful spread per metric, so one extreme airline cannot stretch the scale and tiny differences are not blown up."),
    ("A missing pillar counts as 50.", "Airlines with no 2019 year get a neutral Recovery pillar instead of being averaged on their two best pillars."),
])
nav = "".join(f'<a href="#s{n}">{n}. {esc(t.split(":")[0])}</a>' for n, t in SEC.items()) + '<a href="#concl">Conclusions</a>'
def card(k, v):
    extra = ""
    if k == "corr":
        btns = "".join(f'<button class="pb" onclick="drawPair(\'{a_}\',\'{b_}\')">{esc(LBL[a_])} vs {esc(LBL[b_])} ({r:+.2f})</button>' for a_, b_, r in v["pairs"])
        extra = f'<div class="pbs"><span class="small">Interesting pairs:</span> {btns}</div><div id="pairplot" style="height:440px"></div>'
    return (f'<section class="card" id="c-{k}"><h3>{esc(v["title"])}</h3><p class="how">{esc(v["how"])}</p>'
            f'{v["fig"].to_html(full_html=False, include_plotlyjs=False, div_id="fig-" + k, config=CFG)}{extra}<p class="cap"><b>What it shows.</b> {esc(v["cap"])}</p></section>')
secs = ""
for n, t in SEC.items():
    intro = ""
    if n == 1:
        intro = (f'<div class="box"><h3>How each metric is calculated</h3><p class="small">All metrics use scheduled domestic flights, summed per airline group per month, over the 12 months to {LAST:%B %Y} '
                 f'("previous 12" = the 12 months before). Passenger-km and seat-km are stored in thousands. The last column works each formula through for IndiGo.</p>'
                 f'<div class="tbl"><table><tr><th>Pillar</th><th>Metric</th><th>Formula</th><th>Columns used</th><th>Unit</th><th>Better when</th><th>IndiGo, worked</th></tr>{met_rows}</table></div>'
                 f'<h3>From value to score to index</h3><ol class="small"><li>Each metric is turned into a score: z = (value - median of the scored airlines) / spread, capped at -2 and +2; score = 50 + 25 x z (volatility is flipped, lower is better).</li>'
                 f'<li>A pillar is the average of its metric scores; the index is the average of the three pillars.</li><li>An airline is scored if it flew in all of the last 12 months and averaged at least {MIN_PAX:,} passengers a month. Recovery metrics also need a full, non-tiny 2019 year.</li></ol>'
                 f'<h3>Countermeasures for new airlines</h3><p class="small">New airlines start from a tiny base, which inflates growth and volatility and leaves them without a 2019 yardstick. Six rules keep that from distorting the index:</p><ul class="small">{fl}</ul></div>')
    secs += f'<h2 id="s{n}">{n}. {esc(t)}</h2>{intro}' + "".join(card(k, v) for k, v in FIGS.items() if v["sec"] == n)
concl = "".join(f"<li><b>{esc(t)}</b> {esc(b_)}</li>" for t, b_ in CONCL)
JSDATA = json.dumps(dict(lbl=LBL, unit={k: SHORTU[k] for k in MET}, col={g: gc(g) for g in OBS.g.unique()}, sym={g: gs(g) for g in OBS.g.unique()},
                         order=ORDER + [g for g in OBS.g.unique() if g not in ORDER], corr={a_: {b_: (None if pd.isna(CM.loc[a_, b_]) else round(float(CM.loc[a_, b_]), 2)) for b_ in KEYS} for a_ in KEYS},
                         n={a_: {b_: int(CN.loc[a_, b_]) for b_ in KEYS} for a_ in KEYS},
                         obs={g: {"T": OBS.loc[OBS.g == g, "T"].dt.strftime("%b %Y").tolist(), **{k: [None if pd.isna(x) else round(float(x), 3) for x in OBS.loc[OBS.g == g, k]] for k in KEYS}} for g in OBS.g.unique()},
                         first=list(INTERESTING[0][:2])))
JS = """
const D = %s;
const keyOf = Object.fromEntries(Object.entries(D.lbl).map(([k, v]) => [v, k]));
function drawPair(xk, yk) {
  if (xk === yk) return;
  const tr = D.order.filter(g => D.obs[g]).map(g => {
    const o = D.obs[g], xs = [], ys = [], ts = [];
    o[xk].forEach((x, i) => { if (x !== null && o[yk][i] !== null) { xs.push(x); ys.push(o[yk][i]); ts.push(o.T[i]); } });
    return {x: xs, y: ys, text: ts, name: g, mode: 'markers', type: 'scatter', marker: {color: D.col[g], symbol: D.sym[g], size: 8, line: {color: 'white', width: .5}},
            hovertemplate: g + '<br>%%{text}<br>x %%{x:.1f}, y %%{y:.1f}<extra></extra>'};
  }).filter(t => t.x.length);
  const r = D.corr[xk][yk];
  Plotly.react('pairplot', tr, {template: 'plotly_white', margin: {l: 70, r: 20, t: 50, b: 60}, legend: {orientation: 'h', y: -0.25},
    font: {family: "system-ui, 'Segoe UI', Arial, sans-serif", size: 13, color: '#222'},
    title: {text: D.lbl[xk] + ' vs ' + D.lbl[yk] + ': rank correlation ' + (r === null ? 'n/a' : (r > 0 ? '+' : '') + r.toFixed(2)) + ' (n = ' + D.n[xk][yk] + ')', x: 0, font: {size: 14}},
    xaxis: {title: {text: D.lbl[xk] + ' (' + D.unit[xk] + ')'}}, yaxis: {title: {text: D.lbl[yk] + ' (' + D.unit[yk] + ')'}}}, {displaylogo: false, responsive: true});
  document.getElementById('pairplot').scrollIntoView({behavior: 'smooth', block: 'nearest'});
}
window.addEventListener('load', () => {
  const c = document.getElementById('fig-corr');
  if (c) c.on('plotly_click', e => { const p = e.points[0]; drawPair(keyOf[p.x], keyOf[p.y]); });
  drawPair(D.first[0], D.first[1]); window.scrollTo(0, 0);
});
""" % JSDATA
page = f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Indian airlines after COVID: performance dashboard</title>
<style>
:root{{--ink:#222;--mut:#5f5f5f;--line:#e2e2e2;--bg:#f6f7f9;--acc:#2166AC}}
*{{box-sizing:border-box}} body{{margin:0;font-family:system-ui,'Segoe UI',Arial,sans-serif;color:var(--ink);background:var(--bg);line-height:1.5}} body.grey{{filter:grayscale(1)}}
header{{background:#fff;border-bottom:1px solid var(--line);padding:22px 28px}} h1{{margin:0 0 4px;font-size:26px}} .sub{{color:var(--mut);margin:0;max-width:940px}}
nav{{position:sticky;top:0;z-index:5;background:#fff;border-bottom:1px solid var(--line);padding:8px 28px;display:flex;gap:6px;flex-wrap:wrap;align-items:center}}
nav a,nav button{{color:var(--ink);text-decoration:none;font:inherit;font-size:13px;padding:5px 10px;border-radius:14px;background:var(--bg);border:0;cursor:pointer}}
nav a:hover,nav button:hover{{background:var(--acc);color:#fff}} nav button{{margin-left:auto;border:1px solid var(--line)}}
main{{max-width:1180px;margin:0 auto;padding:18px 16px 60px;overflow-x:hidden}} h2{{margin:36px 0 8px;font-size:21px;scroll-margin-top:60px}}
.kpis{{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px;margin:18px 0}}
.kpi{{background:#fff;border:1px solid var(--line);border-radius:10px;padding:14px 16px}} .kpi .v{{font-size:22px;font-weight:700}} .kpi .l{{font-size:13px}} .kpi .s{{font-size:12px;color:var(--mut)}}
.card{{background:#fff;border:1px solid var(--line);border-radius:10px;padding:14px 16px 12px;margin:14px 0;scroll-margin-top:60px;overflow-x:auto;overflow-y:hidden}} .card h3{{margin:0 0 2px;font-size:17px}}
.how{{margin:0 0 6px;color:var(--mut);font-size:13px}} .cap{{margin:8px 0 0;padding:9px 12px;background:#f1f5f9;border-left:4px solid var(--acc);font-size:14px}}
.tbl{{overflow-x:auto}} table{{border-collapse:collapse;width:100%;font-size:12.5px;background:#fff}} th,td{{border:1px solid var(--line);padding:6px 8px;text-align:left;vertical-align:top}} th{{background:#222;color:#fff}}
.mono{{font-family:ui-monospace,Consolas,monospace;font-size:12px;white-space:nowrap}}
.box{{background:#fff;border:1px solid var(--line);border-radius:10px;padding:6px 20px 14px;margin:14px 0}} li{{margin:6px 0}} .small{{font-size:13px;color:var(--mut)}}
.pbs{{margin:8px 0;display:flex;gap:6px;flex-wrap:wrap;align-items:center}} .pb{{font:inherit;font-size:12.5px;padding:4px 10px;border:1px solid var(--line);border-radius:14px;background:var(--bg);cursor:pointer}} .pb:hover{{background:var(--acc);color:#fff}}
</style><script>{get_plotlyjs()}</script></head><body>
<header><h1>Indian airlines after COVID: performance dashboard</h1>
<p class="sub">Which airlines regained their 2019 passenger numbers and load factors first, and which never did? Ten metrics, a 0 to 100 index with countermeasures for new airlines, the route network from city-pair data, and international flights. Scheduled flights, January 2017 to {LAST:%B %Y}. Source: DGCA monthly statistics.</p></header>
<nav>{nav}<button onclick="document.body.classList.toggle('grey')" title="Show the page in greyscale to check the charts still read without colour">Greyscale check</button></nav><main>
<div class="kpis">{kpi_html}</div>
{secs}
<h2 id="concl">Conclusions</h2><div class="box"><ol>{concl}</ol><h3>Limitations</h3><ul>{"".join(f"<li class='small'>{esc(x)}</li>" for x in LIMITS)}</ul>
<p class="small">Data: DGCA (Government of India) monthly airline traffic statistics, parsed by the team; city-pair and parsed airline tables from Vonter/india-aviation-traffic (ODbL according to the repository README), city-pair file {esc(CITY_FILE.name)}. Charts: Plotly. Page generated {TODAY}.</p></div>
</main><script>{JS}</script></body></html>"""
OUT_FILE = OUT / f"dgca_performance_dashboard_{TODAY}.html"
OUT_FILE.write_text(page, encoding="utf-8")
print(f"saved {OUT_FILE.name}: {OUT_FILE.stat().st_size / 1e6:.1f} MB | {len(FIGS)} charts | works offline")


saved dgca_performance_dashboard_2026-11-05.html: 5.4 MB | 37 charts | works offline
